In [ ]:
from pathlib import Path
import sys
for _root in (Path.cwd(), *Path.cwd().parents):
    if (_root / "src/plotting/regimes.py").is_file():
        sys.path.insert(0, str(_root / "src"))
        break


In [ ]:
# -*- coding: utf-8 -*-
"""
Plot-only / evaluation-only Burgers ICON diagnostics.

This script DOES NOT train. It:
  1. rebuilds the same ICON architecture,
  2. loads saved parameters from a .msgpack checkpoint,
  3. generates Burgers test/context prompts,
  4. computes true Burgers tangent spectra and ICON Jacobian spectra,
  5. saves all figures together in one folder;
  6. makes paper-style figures: ICON spectra plus true-reference spectral errors.

Requested plotting conventions:
  - no pair gain: cos and sin gains are plotted separately;
  - no plot titles;
  - base-state predictions are included in one figure;
  - plotting is centralized at the end: compute_all_diagnostics() then plot_all_figures().
"""

from __future__ import annotations

from collections import namedtuple
from typing import Tuple, Sequence
from functools import partial
import os
import glob

import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
import jax.tree_util as tree
from flax import linen as nn
from flax.linen import MultiHeadDotProductAttention
from flax import serialization
from einshape import jax_einshape as einshape

import models_utils as mu


# =============================================================================
# 0. Configuration: must match training
# =============================================================================

seed = 2026

# Prompt / data settings.
num_pairs = 6
demo_num = 5
n_grid = 100

# IMPORTANT: set this to the n_modes used in your checkpoint/training.
# If your latest checkpoint was trained with n_modes=16, keep 16.
# If it was trained with n_modes=8, change this to 8.
n_modes = 16

fourier_decay = 1.0
bias_scale = 0.0
initial_amp = 0.7
normalize_each_function = False

# Burgers PDE settings.
T_final = 0.25
c_range: Tuple[float, float] = (0.005, 0.05)
n_solver_steps = 800

# Model settings.
MODEL_SIZE = "paper_like"

if MODEL_SIZE == "paper_like":
    model_config = {
        "demo_max_num": num_pairs,
        "index_mode": "learn",
        "transformer": {
            "n_layers": 6,
            "n_heads": 8,
            "head_dim": 32,
            "model_dim": 256,
            "widening_factor": 4,
        },
        "out_dim": 1,
    }
elif MODEL_SIZE == "small":
    model_config = {
        "demo_max_num": num_pairs,
        "index_mode": "learn",
        "transformer": {
            "n_layers": 4,
            "n_heads": 4,
            "head_dim": 32,
            "model_dim": 128,
            "widening_factor": 4,
        },
        "out_dim": 1,
    }
else:
    raise ValueError(f"Unknown MODEL_SIZE={MODEL_SIZE}")

# -------------------------------------------------------------------------
# Checkpoint and figure folder.
# -------------------------------------------------------------------------
ckpt_dir = "../checkpoints/main"
ckpt_path = None
# If ckpt_path is None, the script automatically loads the latest params_step_*.msgpack.
# Otherwise set, e.g.
# ckpt_path = "../checkpoints/main/params_step_200000.msgpack"

save_figs = True
fig_dir = "../figures/current"
os.makedirs(fig_dir, exist_ok=True)

# Diagnostics sizes. Reduce these if JVP/Jacobian is slow.
num_test_prompts = 8
max_mode = 40
fourier_block_max_mode = 16
num_contexts_per_c = 5
base_contexts_per_case = 3


# =============================================================================
# 1. Plot style
# =============================================================================

def set_paper_plot_style():
    plt.rcParams.update({
        "font.size": 12,
        "axes.titlesize": 12,
        "axes.labelsize": 12,
        "legend.fontsize": 10,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "lines.linewidth": 2.0,
        "axes.linewidth": 1.0,
        "grid.alpha": 0.30,
        "figure.dpi": 120,
    })


set_paper_plot_style()


# =============================================================================
# 2. Data containers
# =============================================================================

Data = namedtuple(
    "Data",
    [
        "demo_cond_k", "demo_cond_v", "demo_cond_mask",
        "demo_qoi_k", "demo_qoi_v", "demo_qoi_mask",
        "quest_cond_k", "quest_cond_v", "quest_cond_mask",
        "quest_qoi_k", "quest_qoi_mask",
    ],
)

RawBurgers = namedtuple(
    "RawBurgers",
    ["cond_k", "cond_v", "qoi_k", "qoi_v", "c", "coeff_sin", "coeff_cos", "bias"],
)


# =============================================================================
# 3. Random initial functions and Burgers solver
# =============================================================================

def eval_random_fourier(
    x: jnp.ndarray,
    coeff_sin: jnp.ndarray,
    coeff_cos: jnp.ndarray,
    bias: jnp.ndarray,
) -> jnp.ndarray:
    n_modes_local = coeff_sin.shape[-1]
    modes = jnp.arange(1, n_modes_local + 1, dtype=x.dtype)
    phase = 2.0 * jnp.pi * x[..., None] * modes
    y = jnp.sum(
        coeff_sin[..., None, :] * jnp.sin(phase)
        + coeff_cos[..., None, :] * jnp.cos(phase),
        axis=-1,
    )
    return y + bias[..., None]


def maybe_normalize_pair(cond_v: jnp.ndarray, qoi_v: jnp.ndarray) -> tuple[jnp.ndarray, jnp.ndarray]:
    if not normalize_each_function:
        return cond_v, qoi_v
    mean = jnp.mean(cond_v, axis=2, keepdims=True)
    std = jnp.std(cond_v, axis=2, keepdims=True) + 1e-6
    return (cond_v - mean) / std, (qoi_v - mean) / std


def _burgers_rhs(u: jnp.ndarray, c: jnp.ndarray, k: jnp.ndarray, dealias: jnp.ndarray) -> jnp.ndarray:
    u_hat = jnp.fft.fft(u, axis=-1)
    u2_hat = jnp.fft.fft(u * u, axis=-1) * dealias
    nonlin = -0.5 * jnp.fft.ifft(1j * k * u2_hat, axis=-1).real
    diff = c[:, None] * jnp.fft.ifft(-(k ** 2) * u_hat, axis=-1).real
    return nonlin + diff


def _rk4_step(u: jnp.ndarray, c: jnp.ndarray, dt: float, k: jnp.ndarray, dealias: jnp.ndarray) -> jnp.ndarray:
    k1 = _burgers_rhs(u, c, k, dealias)
    k2 = _burgers_rhs(u + 0.5 * dt * k1, c, k, dealias)
    k3 = _burgers_rhs(u + 0.5 * dt * k2, c, k, dealias)
    k4 = _burgers_rhs(u + dt * k3, c, k, dealias)
    return u + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)


def solve_burgers_batch(u0: jnp.ndarray, c: jnp.ndarray, T: float, nsteps: int) -> jnp.ndarray:
    N = u0.shape[-1]
    dt_step = T / float(nsteps)
    freq = jnp.fft.fftfreq(N, d=1.0 / N)
    k = 2.0 * jnp.pi * freq
    dealias = (jnp.abs(freq) <= (N / 3.0)).astype(jnp.complex64)

    def body(u, _):
        return _rk4_step(u, c, dt_step, k, dealias), None

    uT, _ = jax.lax.scan(body, u0, xs=None, length=nsteps)
    return uT


solve_burgers_batch_jit = jax.jit(solve_burgers_batch, static_argnames=("nsteps",))


# =============================================================================
# 4. Prompt generation
# =============================================================================

def generate_burgers_raw_data(
    key: jax.Array,
    num_operators: int,
    pairs_per_operator: int,
    n_grid: int = 100,
    n_modes: int = 8,
    c_range: Tuple[float, float] = (0.005, 0.05),
    T_final: float = 0.25,
    fourier_decay: float = 1.0,
    bias_scale: float = 0.0,
    initial_amp: float = 0.7,
    n_solver_steps: int = 800,
) -> RawBurgers:
    key_c, key_coef, key_bias = jax.random.split(key, 3)

    c = jax.random.uniform(
        key_c,
        shape=(num_operators,),
        minval=c_range[0],
        maxval=c_range[1],
    )

    modes = jnp.arange(1, n_modes + 1, dtype=jnp.float32)
    scale = initial_amp / (modes ** fourier_decay)

    coeff = jax.random.normal(
        key_coef,
        shape=(num_operators, pairs_per_operator, 2, n_modes),
    ) * scale

    coeff_sin = coeff[:, :, 0, :]
    coeff_cos = coeff[:, :, 1, :]
    bias = bias_scale * jax.random.normal(key_bias, shape=(num_operators, pairs_per_operator))

    x = jnp.linspace(0.0, 1.0, n_grid, endpoint=False, dtype=jnp.float32)
    x_full = jnp.broadcast_to(x, (num_operators, pairs_per_operator, n_grid))

    cond_k = x_full[..., None]
    qoi_k = x_full[..., None]

    cond_v0 = eval_random_fourier(x_full, coeff_sin, coeff_cos, bias)

    flat_u0 = cond_v0.reshape(num_operators * pairs_per_operator, n_grid)
    flat_c = jnp.repeat(c, pairs_per_operator)
    flat_uT = solve_burgers_batch_jit(flat_u0, flat_c, T_final, nsteps=n_solver_steps)

    cond_v = cond_v0[..., None]
    qoi_v = flat_uT.reshape(num_operators, pairs_per_operator, n_grid, 1)
    cond_v, qoi_v = maybe_normalize_pair(cond_v, qoi_v)

    return RawBurgers(
        cond_k=cond_k.astype(jnp.float32),
        cond_v=cond_v.astype(jnp.float32),
        qoi_k=qoi_k.astype(jnp.float32),
        qoi_v=qoi_v.astype(jnp.float32),
        c=c.astype(jnp.float32),
        coeff_sin=coeff_sin.astype(jnp.float32),
        coeff_cos=coeff_cos.astype(jnp.float32),
        bias=bias.astype(jnp.float32),
    )


def build_icon_data(raw: RawBurgers, demo_num: int = 5) -> tuple[Data, jnp.ndarray]:
    bs = raw.cond_k.shape[0]
    cond_len = raw.cond_k.shape[2]
    qoi_len = raw.qoi_k.shape[2]

    data = Data(
        demo_cond_k=raw.cond_k[:, :demo_num],
        demo_cond_v=raw.cond_v[:, :demo_num],
        demo_cond_mask=jnp.ones((bs, demo_num, cond_len), dtype=bool),
        demo_qoi_k=raw.qoi_k[:, :demo_num],
        demo_qoi_v=raw.qoi_v[:, :demo_num],
        demo_qoi_mask=jnp.ones((bs, demo_num, qoi_len), dtype=bool),
        quest_cond_k=raw.cond_k[:, demo_num:demo_num + 1],
        quest_cond_v=raw.cond_v[:, demo_num:demo_num + 1],
        quest_cond_mask=jnp.ones((bs, 1, cond_len), dtype=bool),
        quest_qoi_k=raw.qoi_k[:, demo_num:demo_num + 1],
        quest_qoi_mask=jnp.ones((bs, 1, qoi_len), dtype=bool),
    )
    label = raw.qoi_v[:, demo_num:demo_num + 1]
    return data, label


def make_test_batch(key_seed: int = 1234, eqns: int = 8, c_rng: Tuple[float, float] | None = None):
    if c_rng is None:
        c_rng = c_range
    raw = generate_burgers_raw_data(
        jax.random.PRNGKey(key_seed),
        num_operators=eqns,
        pairs_per_operator=num_pairs,
        n_grid=n_grid,
        n_modes=n_modes,
        c_range=c_rng,
        T_final=T_final,
        fourier_decay=fourier_decay,
        bias_scale=bias_scale,
        initial_amp=initial_amp,
        n_solver_steps=n_solver_steps,
    )
    data, label = build_icon_data(raw, demo_num=demo_num)
    return raw, data, label


def make_fixed_c_batch(c_value: float, key_seed: int = 999, eqns: int = 8):
    eps = 1e-9
    return make_test_batch(key_seed=key_seed, eqns=eqns, c_rng=(c_value, c_value + eps))


def make_burgers_prompts_at_c(
    key: jax.Array,
    c_value: float,
    num_prompts: int,
    pairs_per_prompt: int = num_pairs,
    fixed_query_u0: jnp.ndarray | None = None,
) -> tuple[RawBurgers, Data, jnp.ndarray]:
    raw = generate_burgers_raw_data(
        key,
        num_operators=num_prompts,
        pairs_per_operator=pairs_per_prompt,
        n_grid=n_grid,
        n_modes=n_modes,
        c_range=(float(c_value), float(c_value) + 1e-8),
        T_final=T_final,
        fourier_decay=fourier_decay,
        bias_scale=bias_scale,
        initial_amp=initial_amp,
        n_solver_steps=n_solver_steps,
    )

    if fixed_query_u0 is not None:
        if normalize_each_function:
            raise NotImplementedError(
                "fixed_query_u0 replacement is implemented for normalize_each_function=False."
            )

        uq = jnp.asarray(fixed_query_u0).reshape(n_grid)
        uq_batch = jnp.broadcast_to(uq[None, :], (num_prompts, n_grid))
        c_batch = raw.c
        uT_batch = solve_burgers_batch_jit(
            uq_batch,
            c_batch,
            T_final,
            nsteps=n_solver_steps,
        )

        cond_v = raw.cond_v.at[:, demo_num, :, 0].set(uq_batch)
        qoi_v = raw.qoi_v.at[:, demo_num, :, 0].set(uT_batch)
        raw = raw._replace(cond_v=cond_v, qoi_v=qoi_v)

    data, label = build_icon_data(raw, demo_num=demo_num)
    return raw, data, label


# =============================================================================
# 5. ICON model definition
# =============================================================================

class MLP(nn.Module):
    hidden_dim: int
    out_dim: int
    depth: int

    @nn.compact
    def __call__(self, inputs):
        x = inputs
        for _ in range(self.depth):
            x = nn.Dense(features=self.hidden_dim)(x)
            x = nn.gelu(x)
        x = nn.Dense(features=self.out_dim)(x)
        return x


class SelfAttnTransformer(nn.Module):
    n_layers: int
    n_heads: int
    head_dim: int
    model_dim: int
    widening_factor: int

    @nn.compact
    def __call__(self, inputs, mask=None):
        x = inputs
        for _ in range(self.n_layers):
            attn_block = MultiHeadDotProductAttention(
                num_heads=self.n_heads,
                qkv_features=self.head_dim * self.n_heads,
                out_features=self.model_dim,
            )
            x = attn_block(inputs_q=x, inputs_k=x, inputs_v=x, mask=mask) + x
            x = nn.LayerNorm()(x)
            dense_block = MLP(
                hidden_dim=self.model_dim * self.widening_factor,
                out_dim=self.model_dim,
                depth=1,
            )
            x = dense_block(x) + x
            x = nn.LayerNorm()(x)
        return x


def build_matrices_from_data_shape(data_shape, mode, shot_num_min, return_shape_list=False):
    demo_num_local = data_shape.demo_cond_k[0]
    demo_cond_len = data_shape.demo_cond_k[1]
    demo_qoi_len = data_shape.demo_qoi_k[1]
    quest_cond_len = data_shape.quest_cond_k[1]
    quest_qoi_len = data_shape.quest_qoi_k[1]

    cond_bool_list, qoi_kv_bool_list, qoi_k_bool_list = mu.build_bool_sequence(
        demo_num_local, mode, shot_num_min
    )
    cond_len_raw = [demo_cond_len] * demo_num_local + [quest_cond_len]
    qoi_kv_len_raw = [demo_qoi_len] * demo_num_local + [quest_qoi_len]
    qoi_k_len_raw = [demo_qoi_len] * demo_num_local + [quest_qoi_len]

    cond_len_list = [a * b for a, b in zip(cond_bool_list, cond_len_raw)]
    qoi_kv_len_list = [a * b for a, b in zip(qoi_kv_bool_list, qoi_kv_len_raw)]
    qoi_k_len_list = [a * b for a, b in zip(qoi_k_bool_list, qoi_k_len_raw)]

    basic_mask = mu.build_basic_mask(cond_len_list, qoi_kv_len_list, qoi_k_len_list)
    index_pos = mu.build_index_integer(cond_len_list, qoi_kv_len_list, qoi_k_len_list)
    out_mask = mu.build_out_mask(
        cond_len_list,
        qoi_kv_len_list,
        qoi_k_len_list,
        num_range=(shot_num_min, demo_num_local + 1),
    )

    if return_shape_list:
        return basic_mask, index_pos, out_mask, cond_len_list, qoi_kv_len_list, qoi_k_len_list
    return basic_mask, index_pos, out_mask


class ICON_LM(nn.Module):
    config: dict
    basic_mask: jnp.ndarray
    index_pos: jnp.ndarray
    out_mask: jnp.ndarray

    def setup(self):
        self.pre_projection = nn.Dense(
            self.config["transformer"]["model_dim"], name="pre_projection"
        )
        self.func_pos_embedding = nn.Embed(
            self.config["demo_max_num"] * 3,
            self.config["transformer"]["model_dim"],
            name="func_pos_embedding",
        )
        self.transformer = SelfAttnTransformer(
            **self.config["transformer"], name="transformer"
        )
        self.post_projection = nn.Dense(self.config["out_dim"], name="post_projection")

    def basic_forward(self, data, mode, index_pos, basic_mask):
        demo_num_local = len(data.demo_cond_k)
        cond_bool_list, qoi_kv_bool_list, qoi_k_bool_list = mu.build_bool_sequence(
            demo_num_local, mode=mode, shot_num_min=0
        )
        sequence = mu.build_data_sequence(
            data, cond_bool_list, qoi_kv_bool_list, qoi_k_bool_list
        )
        token_mask = mu.build_data_mask(
            data, cond_bool_list, qoi_kv_bool_list, qoi_k_bool_list
        )

        sequence = self.pre_projection(sequence)
        sequence = sequence + self.func_pos_embedding(index_pos)

        token_mask = einshape("i->ji", token_mask, j=sequence.shape[0])
        attn_mask = token_mask * basic_mask
        attn_mask = einshape("ij->mij", attn_mask, m=self.config["transformer"]["n_heads"])

        sequence = self.transformer(sequence, mask=attn_mask)
        sequence = self.post_projection(sequence)
        return sequence

    def __call__(self, data):
        sequence = self.basic_forward(data, "train", self.index_pos, self.basic_mask)
        return sequence[self.out_mask]

    def predict(self, data):
        data_shape = tree.tree_map(lambda x: x.shape, data)
        basic_mask, index_pos, _ = build_matrices_from_data_shape(
            data_shape, mode="test", shot_num_min=0
        )
        sequence = self.basic_forward(data, "test", index_pos, basic_mask)
        return sequence[-data.quest_qoi_mask.shape[-1]:, :]


# =============================================================================
# 6. Load checkpoint
# =============================================================================

def find_latest_checkpoint(ckpt_dir: str) -> str:
    paths = sorted(glob.glob(os.path.join(ckpt_dir, "params_step_*.msgpack")))
    if not paths:
        raise FileNotFoundError(f"No params_step_*.msgpack found in {ckpt_dir}")
    return paths[-1]


def build_model_and_load_params(ckpt_path: str | None = None):
    if ckpt_path is None:
        ckpt_path = find_latest_checkpoint(ckpt_dir)

    key = jax.random.PRNGKey(seed)
    key_data, key_init = jax.random.split(key)

    # Use a tiny prompt only to initialize model shapes.
    raw, data, _ = make_test_batch(key_seed=int(key_data[0]), eqns=1, c_rng=c_range)
    sample_data = tree.tree_map(lambda x: x[0], data)
    sample_shape = tree.tree_map(lambda x: x.shape, sample_data)
    basic_mask, index_pos, out_mask = build_matrices_from_data_shape(
        sample_shape, mode="train", shot_num_min=0
    )

    icon_model = ICON_LM(
        model_config,
        basic_mask=basic_mask,
        index_pos=index_pos,
        out_mask=out_mask,
    )
    params_template = icon_model.init({"params": key_init}, sample_data)

    with open(ckpt_path, "rb") as f:
        params = serialization.from_bytes(params_template, f.read())

    return icon_model, params, ckpt_path


icon_model, params, loaded_ckpt_path = build_model_and_load_params(ckpt_path)
print("Loaded checkpoint:", loaded_ckpt_path)


def predict_one(params, data_one):
    return icon_model.apply(params, data_one, method=ICON_LM.predict)


predict_batch = jax.jit(jax.vmap(predict_one, in_axes=(None, 0)))


# =============================================================================
# 7. Evaluation and Jacobian utilities
# =============================================================================

def evaluate_prediction(params, data, label):
    pred = predict_batch(params, data)
    target = label[:, 0]
    diff = pred - target
    mse = jnp.mean(diff ** 2)
    rmse = jnp.sqrt(mse)
    rel_l2 = jnp.linalg.norm(diff) / (jnp.linalg.norm(target) + 1e-12)
    return pred, target, mse, rmse, rel_l2


def effective_rank(s, eps=1e-12):
    return (jnp.sum(s) ** 2) / (jnp.sum(s ** 2) + eps)


def fourier_basis(n: int, kmax: int):
    x = jnp.arange(n, dtype=jnp.float32) / n
    cols = [jnp.ones_like(x)]
    k_mode = 1
    while len(cols) < kmax:
        cols.append(jnp.sqrt(2.0) * jnp.cos(2.0 * jnp.pi * k_mode * x))
        if len(cols) < kmax:
            cols.append(jnp.sqrt(2.0) * jnp.sin(2.0 * jnp.pi * k_mode * x))
        k_mode += 1
    B = jnp.stack(cols[:kmax], axis=1)
    Q, _ = jnp.linalg.qr(B)
    return Q[:, :kmax]


def icon_query_jacobian_spectrum(params, data_one, basis_size=41):
    u0_flat = data_one.quest_cond_v.reshape(-1)

    def f_query(u_flat):
        data_new = data_one._replace(
            quest_cond_v=u_flat.reshape(data_one.quest_cond_v.shape)
        )
        y = predict_one(params, data_new)
        return y.reshape(-1)

    J = jax.jacrev(f_query)(u0_flat)
    n_out, n_in = J.shape

    dx_in = 1.0 / n_in
    dx_out = 1.0 / n_out
    J_l2 = jnp.sqrt(dx_out / dx_in) * J

    s_full = jnp.linalg.svd(J_l2, compute_uv=False)

    k_in = min(basis_size, n_in)
    k_out = min(basis_size, n_out)
    Phi = fourier_basis(n_in, k_in)
    Psi = fourier_basis(n_out, k_out)
    A = Psi.T @ J_l2 @ Phi
    s_basis = jnp.linalg.svd(A, compute_uv=False)

    return {
        "J": J,
        "J_l2": J_l2,
        "A_fourier": A,
        "singular_values_full": s_full,
        "singular_values_basis": s_basis,
        "effective_rank_full": effective_rank(s_full),
        "effective_rank_basis": effective_rank(s_basis),
    }


@partial(jax.jit, static_argnames=("max_mode",))
def icon_mode_gain_by_jvp_fast(params, data_one, max_mode=40):
    u0_flat = data_one.quest_cond_v.reshape(-1)
    n = u0_flat.shape[0]

    def f_query(u_flat):
        data_new = data_one._replace(
            quest_cond_v=u_flat.reshape(data_one.quest_cond_v.shape)
        )
        return predict_one(params, data_new).reshape(-1)

    x = jnp.arange(n, dtype=jnp.float32) / n
    ks = jnp.arange(1, max_mode + 1, dtype=jnp.float32)[:, None]
    cos_basis = jnp.sqrt(2.0) * jnp.cos(2.0 * jnp.pi * ks * x[None, :])
    sin_basis = jnp.sqrt(2.0) * jnp.sin(2.0 * jnp.pi * ks * x[None, :])
    tangents = jnp.stack([cos_basis, sin_basis], axis=1).reshape(2 * max_mode, n)

    def one_jvp(v):
        _, y = jax.jvp(f_query, (u0_flat,), (v,))
        return y

    responses = jax.vmap(one_jvp)(tangents)
    gains = jnp.linalg.norm(responses, axis=1) / (jnp.linalg.norm(tangents, axis=1) + 1e-12)
    gains = gains.reshape(max_mode, 2)

    k_values = jnp.arange(1, max_mode + 1, dtype=jnp.float32)
    return jnp.stack([k_values, gains[:, 0], gains[:, 1]], axis=1)


def true_burgers_map_u0(u0_flat: jnp.ndarray, c_value: float | jnp.ndarray) -> jnp.ndarray:
    u0 = u0_flat.reshape(1, n_grid)
    c_arr = jnp.asarray(c_value, dtype=jnp.float32).reshape(1)
    uT = solve_burgers_batch_jit(u0, c_arr, T_final, nsteps=n_solver_steps)
    return uT.reshape(-1)


def true_burgers_jacobian_spectrum_u0(u0_flat, c_value, basis_size=41):
    f = lambda u: true_burgers_map_u0(u, c_value)
    J = jax.jacrev(f)(u0_flat)
    n_out, n_in = J.shape

    dx_in = 1.0 / n_in
    dx_out = 1.0 / n_out
    J_l2 = jnp.sqrt(dx_out / dx_in) * J

    s_full = jnp.linalg.svd(J_l2, compute_uv=False)

    k_in = min(basis_size, n_in)
    k_out = min(basis_size, n_out)
    Phi = fourier_basis(n_in, k_in)
    Psi = fourier_basis(n_out, k_out)
    A = Psi.T @ J_l2 @ Phi
    s_basis = jnp.linalg.svd(A, compute_uv=False)

    return {
        "J": J,
        "J_l2": J_l2,
        "A_fourier": A,
        "singular_values_full": s_full,
        "singular_values_basis": s_basis,
        "effective_rank_full": effective_rank(s_full),
        "effective_rank_basis": effective_rank(s_basis),
    }


@partial(jax.jit, static_argnames=("max_mode",))
def true_mode_gain_by_jvp_u0_fast(u0_flat, c_value, max_mode=40):
    u0_flat = jnp.asarray(u0_flat).reshape(-1)
    n = u0_flat.shape[0]
    f = lambda u: true_burgers_map_u0(u, c_value)

    x = jnp.arange(n, dtype=jnp.float32) / n
    ks = jnp.arange(1, max_mode + 1, dtype=jnp.float32)[:, None]
    cos_basis = jnp.sqrt(2.0) * jnp.cos(2.0 * jnp.pi * ks * x[None, :])
    sin_basis = jnp.sqrt(2.0) * jnp.sin(2.0 * jnp.pi * ks * x[None, :])
    tangents = jnp.stack([cos_basis, sin_basis], axis=1).reshape(2 * max_mode, n)

    def one_jvp(v):
        _, y = jax.jvp(f, (u0_flat,), (v,))
        return y

    responses = jax.vmap(one_jvp)(tangents)
    gains = jnp.linalg.norm(responses, axis=1) / (jnp.linalg.norm(tangents, axis=1) + 1e-12)
    gains = gains.reshape(max_mode, 2)

    k_values = jnp.arange(1, max_mode + 1, dtype=jnp.float32)
    return jnp.stack([k_values, gains[:, 0], gains[:, 1]], axis=1)


def true_burgers_sensitivity_c(u0_flat, c_value):
    u0_flat = jnp.asarray(u0_flat).reshape(-1)

    def f_c(c_scalar):
        return true_burgers_map_u0(u0_flat, c_scalar)

    du_dc = jax.jacfwd(f_c)(jnp.asarray(c_value, dtype=jnp.float32))
    return du_dc.reshape(-1)


def fourier_coefficients_of_function(y, max_mode=40):
    y = jnp.asarray(y).reshape(-1)
    n = y.shape[0]
    x = jnp.arange(n, dtype=jnp.float32) / n
    rows = []
    for k_mode in range(1, max_mode + 1):
        phi_cos = jnp.sqrt(2.0) * jnp.cos(2.0 * jnp.pi * k_mode * x)
        phi_sin = jnp.sqrt(2.0) * jnp.sin(2.0 * jnp.pi * k_mode * x)
        cc = jnp.dot(y, phi_cos) / (jnp.dot(phi_cos, phi_cos) + 1e-12)
        ss = jnp.dot(y, phi_sin) / (jnp.dot(phi_sin, phi_sin) + 1e-12)
        rows.append([k_mode, cc, ss])
    return jnp.asarray(rows)


# =============================================================================
# 8. Base states
# =============================================================================

def make_burgers_base_state(kind: str, n_grid: int = 100) -> jnp.ndarray:
    x = jnp.linspace(0.0, 1.0, n_grid, endpoint=False, dtype=jnp.float32)

    if kind == "smooth":
        u = jnp.sin(2.0 * jnp.pi * x)
    elif kind == "steep":
        width = 0.22
        u = jnp.tanh(jnp.sin(2.0 * jnp.pi * x) / width)
    elif kind == "near_shock":
        width = 0.08
        u = jnp.tanh(jnp.sin(2.0 * jnp.pi * x) / width)
    else:
        raise ValueError(f"Unknown base state kind: {kind}")

    u = u - jnp.mean(u)
    max_abs = jnp.max(jnp.abs(u)) + 1e-12
    u = initial_amp * u / max_abs
    return u[None, :, None].astype(jnp.float32)


def make_prompt_with_fixed_c_and_uq(c_value: float, fixed_uq: jnp.ndarray, key_seed: int):
    raw, data, label = make_fixed_c_batch(c_value, key_seed=key_seed, eqns=1)
    data_one = tree.tree_map(lambda x: x[0], data)
    data_one = data_one._replace(quest_cond_v=fixed_uq)
    return data_one, raw


# =============================================================================
# 9. Cos/sin Fourier blocks
# =============================================================================

def fourier_cos_sin_bases(n: int, max_mode: int):
    x = jnp.arange(n, dtype=jnp.float32) / n
    C, S = [], []
    for k_mode in range(1, max_mode + 1):
        c_col = jnp.sqrt(2.0) * jnp.cos(2.0 * jnp.pi * k_mode * x)
        s_col = jnp.sqrt(2.0) * jnp.sin(2.0 * jnp.pi * k_mode * x)
        c_col = c_col / (jnp.linalg.norm(c_col) + 1e-12)
        s_col = s_col / (jnp.linalg.norm(s_col) + 1e-12)
        C.append(c_col)
        S.append(s_col)
    return jnp.stack(C, axis=1), jnp.stack(S, axis=1)


def fourier_block_projection(J_l2: jnp.ndarray, max_mode: int = 16):
    n_out, n_in = J_l2.shape
    C_in, S_in = fourier_cos_sin_bases(n_in, max_mode)
    C_out, S_out = fourier_cos_sin_bases(n_out, max_mode)

    return {
        "cc": C_out.T @ J_l2 @ C_in,
        "cs": C_out.T @ J_l2 @ S_in,
        "sc": S_out.T @ J_l2 @ C_in,
        "ss": S_out.T @ J_l2 @ S_in,
    }


# =============================================================================
# 10. Compute all diagnostics first
# =============================================================================

def compute_context_spectrum(fixed_uq, c_values, num_contexts_per_c=5, max_mode=40, key_seed=777):
    key0 = jax.random.PRNGKey(key_seed)

    icon_mean = []
    icon_std = []
    true_gains = []

    for i, c_value in enumerate(c_values):
        key_i = jax.random.fold_in(key0, i)
        raw_i, data_i, label_i = make_burgers_prompts_at_c(
            key_i,
            c_value=float(c_value),
            num_prompts=num_contexts_per_c,
            pairs_per_prompt=num_pairs,
            fixed_query_u0=fixed_uq,
        )

        icon_gains_c = []
        for j in range(num_contexts_per_c):
            data_one = tree.tree_map(lambda x: x[j], data_i)
            g_icon = icon_mode_gain_by_jvp_fast(params, data_one, max_mode=max_mode)
            icon_gains_c.append(np.asarray(g_icon[:, 1:3]))

        icon_gains_c = np.stack(icon_gains_c, axis=0)
        icon_mean.append(icon_gains_c.mean(axis=0))
        icon_std.append(icon_gains_c.std(axis=0))

        g_true = true_mode_gain_by_jvp_u0_fast(
            jnp.asarray(fixed_uq).reshape(-1),
            float(c_value),
            max_mode=max_mode,
        )
        true_gains.append(np.asarray(g_true[:, 1:3]))

    return {
        "k": np.arange(1, max_mode + 1),
        "c_values": np.asarray(c_values, dtype=float),
        "icon_mean": np.stack(icon_mean, axis=0),  # [C, K, 2]
        "icon_std": np.stack(icon_std, axis=0),
        "true": np.stack(true_gains, axis=0),      # [C, K, 2]
    }


def compute_base_state_predictions(base_kinds, c_value=0.02, key_seed=444):
    rows = []
    for i, kind in enumerate(base_kinds):
        uq = make_burgers_base_state(kind, n_grid=n_grid)
        raw, data, label = make_burgers_prompts_at_c(
            jax.random.PRNGKey(key_seed + i),
            c_value=float(c_value),
            num_prompts=1,
            pairs_per_prompt=num_pairs,
            fixed_query_u0=uq,
        )
        pred = predict_batch(params, data)
        rows.append({
            "kind": kind,
            "x": np.asarray(data.quest_qoi_k[0, 0, :, 0]),
            "u0": np.asarray(data.quest_cond_v[0, 0, :, 0]),
            "true": np.asarray(label[0, 0, :, 0]),
            "pred": np.asarray(pred[0, :, 0]),
            "c": float(raw.c[0]),
        })
    return rows


def compute_base_state_spectra(base_kinds, c_value=0.02, contexts_per_base=3, max_mode=40, key_seed=13579):
    out = {}
    k = np.arange(1, max_mode + 1)

    for bi, kind in enumerate(base_kinds):
        uq = make_burgers_base_state(kind, n_grid=n_grid)

        true_gain = np.asarray(
            true_mode_gain_by_jvp_u0_fast(
                jnp.asarray(uq).reshape(-1),
                float(c_value),
                max_mode=max_mode,
            )
        )[:, 1:3]

        icon_list = []
        for r in range(contexts_per_base):
            data_one, _ = make_prompt_with_fixed_c_and_uq(
                float(c_value),
                uq,
                key_seed=key_seed + 1000 * bi + r,
            )
            g_icon = icon_mode_gain_by_jvp_fast(params, data_one, max_mode=max_mode)
            icon_list.append(np.asarray(g_icon[:, 1:3]))

        icon_arr = np.stack(icon_list, axis=0)

        out[kind] = {
            "k": k,
            "true": true_gain,                  # [K, 2] cos/sin
            "icon_mean": icon_arr.mean(axis=0), # [K, 2]
            "icon_std": icon_arr.std(axis=0),
        }

    return out


def compute_base_state_viscosity_grid(
    base_kinds,
    c_list,
    contexts_per_case=2,
    max_mode=30,
    key_seed=97531,
):
    results = {}
    k = np.arange(1, max_mode + 1)

    for bi, kind in enumerate(base_kinds):
        uq = make_burgers_base_state(kind, n_grid=n_grid)
        results[kind] = {}

        for ci, c_value in enumerate(c_list):
            true_gain = np.asarray(
                true_mode_gain_by_jvp_u0_fast(
                    jnp.asarray(uq).reshape(-1),
                    float(c_value),
                    max_mode=max_mode,
                )
            )[:, 1:3]

            icon_list = []
            for r in range(contexts_per_case):
                data_one, _ = make_prompt_with_fixed_c_and_uq(
                    float(c_value),
                    uq,
                    key_seed=key_seed + 10000 * bi + 1000 * ci + r,
                )
                g_icon = icon_mode_gain_by_jvp_fast(params, data_one, max_mode=max_mode)
                icon_list.append(np.asarray(g_icon[:, 1:3]))

            icon_arr = np.stack(icon_list, axis=0)

            rel_err = np.linalg.norm(icon_arr.mean(axis=0) - true_gain) / (
                np.linalg.norm(true_gain) + 1e-12
            )

            results[kind][float(c_value)] = {
                "k": k,
                "true": true_gain,
                "icon_mean": icon_arr.mean(axis=0),
                "icon_std": icon_arr.std(axis=0),
                "rel_err": rel_err,
            }

    return results


def compute_split_fourier_blocks(data_one, c_value, max_mode=16):
    u0_flat = data_one.quest_cond_v.reshape(-1)
    spec_true = true_burgers_jacobian_spectrum_u0(u0_flat, c_value, basis_size=2 * max_mode + 1)
    spec_icon = icon_query_jacobian_spectrum(params, data_one, basis_size=2 * max_mode + 1)

    blocks_true = fourier_block_projection(spec_true["J_l2"], max_mode=max_mode)
    blocks_icon = fourier_block_projection(spec_icon["J_l2"], max_mode=max_mode)
    blocks_err = {k: blocks_icon[k] - blocks_true[k] for k in blocks_true}

    return {
        "true": {k: np.asarray(v) for k, v in blocks_true.items()},
        "icon": {k: np.asarray(v) for k, v in blocks_icon.items()},
        "error": {k: np.asarray(v) for k, v in blocks_err.items()},
    }


def compute_all_diagnostics():
    print("\nGenerating diagnostic prompts...")
    raw_test, test_data, test_label = make_burgers_prompts_at_c(
        jax.random.PRNGKey(1234),
        c_value=0.007,
        num_prompts=num_test_prompts,
        pairs_per_prompt=num_pairs,
        fixed_query_u0=None,
    )
    pred, target, mse, rmse, rel_l2 = evaluate_prediction(params, test_data, test_label)

    data_one = tree.tree_map(lambda x: x[0], test_data)
    c0 = float(raw_test.c[0])
    fixed_uq = np.asarray(data_one.quest_cond_v[0, :, 0])

    print(f"diagnostic c={c0:.4f}, mse={float(mse):.6e}, rel_l2={float(rel_l2):.6e}")

    print("Computing true vs ICON query spectrum...")
    g_icon = np.asarray(icon_mode_gain_by_jvp_fast(params, data_one, max_mode=max_mode))
    g_true = np.asarray(true_mode_gain_by_jvp_u0_fast(data_one.quest_cond_v.reshape(-1), c0, max_mode=max_mode))

    print("Computing Fourier-projected singular values and physical-space u0 Jacobians...")
    spec_true = true_burgers_jacobian_spectrum_u0(data_one.quest_cond_v.reshape(-1), c0, basis_size=41)
    spec_icon = icon_query_jacobian_spectrum(params, data_one, basis_size=41)

    print("Computing viscosity sensitivity...")
    du_dc = true_burgers_sensitivity_c(fixed_uq, c0)
    dc_coeffs = fourier_coefficients_of_function(du_dc, max_mode=max_mode)

    print("Computing split Fourier blocks...")
    split_blocks = compute_split_fourier_blocks(data_one, c0, max_mode=fourier_block_max_mode)

    print("Computing OOD context spectra...")
    c_values_ood = [0.002, 0.0035, 0.005, 0.01, 0.02, 0.035, 0.05, 0.07, 0.10]
    context = compute_context_spectrum(
        fixed_uq=fixed_uq,
        c_values=c_values_ood,
        num_contexts_per_c=num_contexts_per_c,
        max_mode=max_mode,
        key_seed=2028,
    )

    print("Computing base-state predictions and spectra...")
    base_kinds = ("smooth", "steep", "near_shock")
    base_predictions = compute_base_state_predictions(base_kinds, c_value=0.02, key_seed=444)
    base_spectra = compute_base_state_spectra(
        base_kinds,
        c_value=0.007,
        contexts_per_base=base_contexts_per_case,
        max_mode=max_mode,
    )
    base_grid = compute_base_state_viscosity_grid(
        base_kinds,
        c_list=(0.005, 0.02, 0.05),
        contexts_per_case=2,
        max_mode=30,
    )

    results = {
        "raw_test": raw_test,
        "test_data": test_data,
        "test_label": test_label,
        "pred": np.asarray(pred),
        "target": np.asarray(target),
        "mse": float(mse),
        "rel_l2": float(rel_l2),
        "c0": c0,
        "query_g_true": g_true,
        "query_g_icon": g_icon,
        "singular_true": np.asarray(spec_true["singular_values_basis"]),
        "singular_icon": np.asarray(spec_icon["singular_values_basis"]),
        "J_true_l2": np.asarray(spec_true["J_l2"]),
        "J_icon_l2": np.asarray(spec_icon["J_l2"]),
        "du_dc": np.asarray(du_dc),
        "dc_coeffs": np.asarray(dc_coeffs),
        "split_blocks": split_blocks,
        "context": context,
        "base_predictions": base_predictions,
        "base_spectra": base_spectra,
        "base_grid": base_grid,
    }
    return results


# =============================================================================
# 11. Plot all figures at the end
# =============================================================================

def save_or_show(path=None):
    plt.tight_layout()
    if save_figs and path is not None:
        plt.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()


def plot_prediction_example(results):
    data = results["test_data"]
    label = results["test_label"]
    pred = results["pred"]
    raw = results["raw_test"]

    i = 0
    x = np.asarray(data.quest_qoi_k[i, 0, :, 0])
    true_i = np.asarray(label[i, 0, :, 0])
    pred_i = np.asarray(pred[i, :, 0])

    plt.figure(figsize=(6, 4))
    plt.plot(x, true_i, "k-", label="true")
    plt.plot(x, pred_i, "r--", label="ICON")
    plt.xlabel(r"$x$")
    plt.ylabel(r"$u(T,x)$")
    plt.grid(True)
    plt.legend()
    save_or_show(os.path.join(fig_dir, "prediction_example.png"))


def _gain_curve_errors(icon_gain: np.ndarray, true_gain: np.ndarray, eps: float = 1e-12):
    """
    Return curve-level relative errors for cos and sin gains.
    icon_gain, true_gain: arrays with columns [k, cos, sin] or [K, 2].
    """
    if icon_gain.shape[-1] == 3:
        icon_cs = icon_gain[:, 1:3]
        true_cs = true_gain[:, 1:3]
    else:
        icon_cs = icon_gain
        true_cs = true_gain
    cos_err = np.linalg.norm(icon_cs[:, 0] - true_cs[:, 0]) / (np.linalg.norm(true_cs[:, 0]) + eps)
    sin_err = np.linalg.norm(icon_cs[:, 1] - true_cs[:, 1]) / (np.linalg.norm(true_cs[:, 1]) + eps)
    return float(cos_err), float(sin_err)


def plot_query_spectrum_with_true_error(results, pointwise_error: str = "absolute"):
    """
    Paper-style replacement for the old true-vs-ICON query-gain plot.

    Top row: ICON Jacobian gain for cos/sin Fourier perturbations.
    Bottom row: error against the true Burgers tangent spectrum.

    We do not plot the true spectrum itself here; it is used only as the
    reference for the error curves. This keeps the main figure compact while
    quantitatively validating the ICON spectrum.
    """
    g_true = results["query_g_true"]
    g_icon = results["query_g_icon"]
    k = g_true[:, 0]

    if pointwise_error == "relative":
        denom_cos = np.maximum(np.abs(g_true[:, 1]), 1e-6)
        denom_sin = np.maximum(np.abs(g_true[:, 2]), 1e-6)
        err_cos = np.abs(g_icon[:, 1] - g_true[:, 1]) / denom_cos
        err_sin = np.abs(g_icon[:, 2] - g_true[:, 2]) / denom_sin
        err_ylabel = "relative gain error"
        err_scale = "log"
    else:
        err_cos = np.abs(g_icon[:, 1] - g_true[:, 1])
        err_sin = np.abs(g_icon[:, 2] - g_true[:, 2])
        err_ylabel = "absolute gain error"
        err_scale = "linear"

    cos_curve_err, sin_curve_err = _gain_curve_errors(g_icon, g_true)

    fig, axes = plt.subplots(2, 2, figsize=(10, 6.5), sharex=True)

    # Top row: ICON-only query spectrum.
    axes[0, 0].plot(k, g_icon[:, 1], "o-", label="ICON")
    axes[0, 1].plot(k, g_icon[:, 2], "s-", label="ICON")

    # Bottom row: true-reference error.
    axes[1, 0].plot(k, err_cos, "o-", label=rf"curve err = {cos_curve_err:.2e}")
    axes[1, 1].plot(k, err_sin, "s-", label=rf"curve err = {sin_curve_err:.2e}")

    for ax in axes.ravel():
        ax.axvline(n_modes, linestyle="--", color="k")
        ax.grid(True, which="both")

    axes[0, 0].set_ylabel("ICON Jacobian gain")
    axes[1, 0].set_ylabel(err_ylabel)
    axes[1, 0].set_xlabel(r"Fourier mode $k$")
    axes[1, 1].set_xlabel(r"Fourier mode $k$")

    axes[0, 0].text(0.04, 0.90, "cos", transform=axes[0, 0].transAxes,
                    ha="left", va="top", bbox=dict(facecolor="white", alpha=0.75, edgecolor="none"))
    axes[0, 1].text(0.04, 0.90, "sin", transform=axes[0, 1].transAxes,
                    ha="left", va="top", bbox=dict(facecolor="white", alpha=0.75, edgecolor="none"))
    axes[1, 0].text(0.04, 0.90, "cos", transform=axes[1, 0].transAxes,
                    ha="left", va="top", bbox=dict(facecolor="white", alpha=0.75, edgecolor="none"))
    axes[1, 1].text(0.04, 0.90, "sin", transform=axes[1, 1].transAxes,
                    ha="left", va="top", bbox=dict(facecolor="white", alpha=0.75, edgecolor="none"))

    if err_scale == "log":
        axes[1, 0].set_yscale("log")
        axes[1, 1].set_yscale("log")

    axes[0, 1].legend()
    axes[1, 0].legend()
    axes[1, 1].legend()
    save_or_show(os.path.join(fig_dir, "burgers_query_spectrum_icon_with_true_error.png"))

def plot_singular_values(results):
    s_true = results["singular_true"]
    s_icon = results["singular_icon"]

    plt.figure(figsize=(6, 4))
    plt.semilogy(s_true / (s_true[0] + 1e-12), "k-o", label="true")
    plt.semilogy(s_icon / (s_icon[0] + 1e-12), "r--s", label="ICON")
    plt.xlabel("index")
    plt.ylabel("normalized singular value")
    plt.grid(True, which="both")
    plt.legend()
    save_or_show(os.path.join(fig_dir, "true_vs_icon_singular_values.png"))



def plot_fixed_c_u0_jacobian_physical(results):
    """
    Plot the physical-grid Jacobian with respect to the query initial condition
    at fixed viscosity c:
        J_true = D_{u0} G_c^T(u0),
        J_ICON = D_{u0} G_C^ICON(u0).

    This is the direct fixed-c, u0-Jacobian diagnostic in physical space.
    Rows are output grid points x_out and columns are input grid points x_in.
    """
    J_true = results["J_true_l2"]
    J_icon = results["J_icon_l2"]
    J_err = J_icon - J_true

    vmax = max(float(np.max(np.abs(J_true))), float(np.max(np.abs(J_icon))), 1e-12)
    err_vmax = max(float(np.max(np.abs(J_err))), 1e-12)

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), constrained_layout=True)
    panels = [
        (J_true, "true", vmax),
        (J_icon, "ICON", vmax),
        (J_err, "ICON - true", err_vmax),
    ]
    for ax, (M, label, vm) in zip(axes, panels):
        im = ax.imshow(
            M,
            origin="lower",
            aspect="auto",
            cmap="coolwarm",
            vmin=-vm,
            vmax=vm,
            extent=[0, 1, 0, 1],
        )
        ax.set_xlabel(r"input location $x_{\mathrm{in}}$")
        ax.set_ylabel(r"output location $x_{\mathrm{out}}$")
        ax.text(
            0.03, 0.94, label,
            transform=ax.transAxes,
            ha="left", va="top",
            bbox=dict(facecolor="white", alpha=0.78, edgecolor="none"),
        )
        fig.colorbar(im, ax=ax, shrink=0.82)

    if save_figs:
        plt.savefig(
            os.path.join(fig_dir, "fixed_c_u0_jacobian_physical_true_icon_error.png"),
            dpi=220,
            bbox_inches="tight",
        )
    plt.show()

def plot_c_sensitivity(results):
    coeffs = results["dc_coeffs"]
    k = coeffs[:, 0]

    plt.figure(figsize=(6, 4))
    plt.semilogy(k, np.abs(coeffs[:, 1]), "o-", label=r"cos coefficient")
    plt.semilogy(k, np.abs(coeffs[:, 2]), "s-", label=r"sin coefficient")
    plt.xlabel(r"Fourier mode $k$")
    plt.ylabel(r"$|\partial_c G_c^T(u_0)|$ coefficient")
    plt.grid(True, which="both")
    plt.legend()
    save_or_show(os.path.join(fig_dir, "true_c_sensitivity_spectrum_cos_sin.png"))


def plot_fourier_blocks_grid(blocks_dict, filename, vmax=None, error=False):
    mats = [
        ("cc", blocks_dict["cc"]),
        ("cs", blocks_dict["cs"]),
        ("sc", blocks_dict["sc"]),
        ("ss", blocks_dict["ss"]),
    ]
    if vmax is None:
        vmax = max(float(np.max(np.abs(M))) for _, M in mats) + 1e-12

    fig, axes = plt.subplots(2, 2, figsize=(8, 7), constrained_layout=True)
    for ax, (name, M) in zip(axes.ravel(), mats):
        im = ax.imshow(
            M,
            origin="lower",
            aspect="auto",
            cmap="coolwarm",
            vmin=-vmax,
            vmax=vmax,
        )
        ax.set_xlabel("input mode")
        ax.set_ylabel("output mode")
        ax.text(
            0.03, 0.94, name,
            transform=ax.transAxes,
            ha="left", va="top",
            fontsize=11,
            bbox=dict(facecolor="white", alpha=0.75, edgecolor="none"),
        )
        fig.colorbar(im, ax=ax, shrink=0.8)
    if save_figs:
        plt.savefig(os.path.join(fig_dir, filename), dpi=220, bbox_inches="tight")
    plt.show()


def plot_split_blocks(results):
    blocks = results["split_blocks"]
    vmax = max(
        max(float(np.max(np.abs(blocks["true"][k]))) for k in blocks["true"]),
        max(float(np.max(np.abs(blocks["icon"][k]))) for k in blocks["icon"]),
        1e-12,
    )
    err_vmax = max(float(np.max(np.abs(blocks["error"][k]))) for k in blocks["error"]) + 1e-12

    plot_fourier_blocks_grid(blocks["true"], "split_fourier_blocks_true.png", vmax=vmax)
    plot_fourier_blocks_grid(blocks["icon"], "split_fourier_blocks_icon.png", vmax=vmax)
    plot_fourier_blocks_grid(blocks["error"], "split_fourier_blocks_error.png", vmax=err_vmax, error=True)


def plot_context_spectrum(results):
    """
    Paper-style context-spectrum figure.

    Top row: ICON context-conditioned query Jacobian spectrum for different c.
    Bottom row: curve-level relative error against the true Burgers tangent
    spectrum. We do not save a separate true-spectrum figure by default.
    """
    from plotting.regimes import style_parameter_curves, style_parameter_sweep, add_regime_key, comparison_filename
    context = results["context"]
    k = context["k"]
    c_values = context["c_values"]
    icon_mean = context["icon_mean"]
    true = context["true"]

    cos_err = np.linalg.norm(icon_mean[:, :, 0] - true[:, :, 0], axis=1) / (
        np.linalg.norm(true[:, :, 0], axis=1) + 1e-12
    )
    sin_err = np.linalg.norm(icon_mean[:, :, 1] - true[:, :, 1], axis=1) / (
        np.linalg.norm(true[:, :, 1], axis=1) + 1e-12
    )

    fig = plt.figure(figsize=(11, 7.2))
    gs = fig.add_gridspec(2, 2, height_ratios=[2.0, 1.0], hspace=0.32, wspace=0.12)
    ax_cos = fig.add_subplot(gs[0, 0])
    ax_sin = fig.add_subplot(gs[0, 1], sharey=ax_cos)
    ax_err = fig.add_subplot(gs[1, :])

    for i, c_value in enumerate(c_values):
        ax_cos.plot(k, icon_mean[i, :, 0], "o-", label=f"{c_value:g}")
        ax_sin.plot(k, icon_mean[i, :, 1], "o-", label=f"{c_value:g}")

    for ax in [ax_cos, ax_sin]:
        ax.axvline(n_modes, linestyle="--", color="k")
        ax.set_xlabel(r"Fourier mode $k$")
        ax.grid(True)

    ax_cos.set_ylabel("ICON Jacobian gain")
    ax_cos.text(0.04, 0.90, "cos", transform=ax_cos.transAxes,
                ha="left", va="top", bbox=dict(facecolor="white", alpha=0.75, edgecolor="none"))
    ax_sin.text(0.04, 0.90, "sin", transform=ax_sin.transAxes,
                ha="left", va="top", bbox=dict(facecolor="white", alpha=0.75, edgecolor="none"))
    style_parameter_curves([ax_cos, ax_sin], c_values, "burgers")
    ax_sin.legend(title=r"$c$", ncol=2)

    ax_err.semilogy(c_values, cos_err, "o-", label="cos")
    ax_err.semilogy(c_values, sin_err, "s-", label="sin")
    ax_err.axvspan(c_range[0], c_range[1], alpha=0.18, label="training range")
    ax_err.set_xlabel(r"hidden viscosity $c$")
    ax_err.set_ylabel("relative spectral error")
    ax_err.grid(True, which="both")
    style_parameter_sweep(ax_err, "burgers")
    ax_err.legend(ncol=3)
    add_regime_key(fig)

    save_or_show(comparison_filename(os.path.join(fig_dir, "burgers_context_spectrum_icon_with_error_vs_c.png")))

    return cos_err, sin_err

def plot_base_state_predictions(results):
    rows = results["base_predictions"]
    fig, axes = plt.subplots(len(rows), 2, figsize=(9, 3.0 * len(rows)), sharex=True)

    for i, row in enumerate(rows):
        axes[i, 0].plot(row["x"], row["u0"], "b-")
        axes[i, 0].set_ylabel(r"$u_0(x)$")
        axes[i, 0].grid(True)
        axes[i, 0].text(
            0.03, 0.88, row["kind"],
            transform=axes[i, 0].transAxes,
            ha="left", va="top",
            bbox=dict(facecolor="white", alpha=0.75, edgecolor="none"),
        )

        axes[i, 1].plot(row["x"], row["true"], "k-", label="true")
        axes[i, 1].plot(row["x"], row["pred"], "r--", label="ICON")
        axes[i, 1].set_ylabel(r"$u(T,x)$")
        axes[i, 1].grid(True)
        if i == 0:
            axes[i, 1].legend()

    axes[-1, 0].set_xlabel(r"$x$")
    axes[-1, 1].set_xlabel(r"$x$")
    save_or_show(os.path.join(fig_dir, "burgers_base_state_predictions.png"))


def plot_base_state_spectra(results):
    base_spectra = results["base_spectra"]
    kinds = list(base_spectra.keys())

    fig, axes = plt.subplots(len(kinds), 2, figsize=(10, 3.0 * len(kinds)), sharex=True, sharey=True)
    for i, kind in enumerate(kinds):
        item = base_spectra[kind]
        k = item["k"]

        # cos
        axes[i, 0].plot(k, item["true"][:, 0], "k-o", label="true")
        axes[i, 0].plot(k, item["icon_mean"][:, 0], "r--s", label="ICON")
        axes[i, 0].axvline(n_modes, linestyle="--", color="gray")
        axes[i, 0].grid(True)
        axes[i, 0].set_ylabel("gain")
        axes[i, 0].text(
            0.03, 0.88, f"{kind}, cos",
            transform=axes[i, 0].transAxes,
            ha="left", va="top",
            bbox=dict(facecolor="white", alpha=0.75, edgecolor="none"),
        )

        # sin
        axes[i, 1].plot(k, item["true"][:, 1], "k-o", label="true")
        axes[i, 1].plot(k, item["icon_mean"][:, 1], "r--s", label="ICON")
        axes[i, 1].axvline(n_modes, linestyle="--", color="gray")
        axes[i, 1].grid(True)
        axes[i, 1].text(
            0.03, 0.88, f"{kind}, sin",
            transform=axes[i, 1].transAxes,
            ha="left", va="top",
            bbox=dict(facecolor="white", alpha=0.75, edgecolor="none"),
        )

        if i == 0:
            axes[i, 0].legend()
            axes[i, 1].legend()

    axes[-1, 0].set_xlabel(r"Fourier mode $k$")
    axes[-1, 1].set_xlabel(r"Fourier mode $k$")
    save_or_show(os.path.join(fig_dir, "burgers_true_vs_icon_base_state_spectra_cos_sin.png"))


def plot_base_state_viscosity_grid(results):
    base_grid = results["base_grid"]
    kinds = list(base_grid.keys())
    c_list = list(next(iter(base_grid.values())).keys())

    # One figure per cos/sin component.
    for comp, comp_name in enumerate(["cos", "sin"]):
        fig, axes = plt.subplots(1, len(kinds), figsize=(5 * len(kinds), 4), sharey=True)
        if len(kinds) == 1:
            axes = [axes]

        for i, kind in enumerate(kinds):
            ax = axes[i]
            for c_value in c_list:
                item = base_grid[kind][c_value]
                k = item["k"]
                ax.plot(k, item["true"][:, comp], "k-", alpha=0.55)
                ax.plot(k, item["icon_mean"][:, comp], "o--", label=f"{c_value:g}")
            ax.axvline(n_modes, linestyle="--", color="gray")
            ax.grid(True)
            ax.set_xlabel(r"Fourier mode $k$")
            ax.text(
                0.03, 0.88, kind,
                transform=ax.transAxes,
                ha="left", va="top",
                bbox=dict(facecolor="white", alpha=0.75, edgecolor="none"),
            )
        axes[0].set_ylabel(f"{comp_name} gain")
        axes[-1].legend(title=r"$c$")
        save_or_show(os.path.join(fig_dir, f"burgers_true_vs_icon_base_state_viscosity_grid_{comp_name}.png"))


def save_numeric_results(results, context_errors=None):
    context = results["context"]
    base_kinds = list(results["base_spectra"].keys())

    save_dict = {
        "c0": np.asarray(results["c0"]),
        "mse": np.asarray(results["mse"]),
        "rel_l2": np.asarray(results["rel_l2"]),
        "query_g_true": results["query_g_true"],
        "query_g_icon": results["query_g_icon"],
        "singular_true": results["singular_true"],
        "singular_icon": results["singular_icon"],
        "J_true_l2": results["J_true_l2"],
        "J_icon_l2": results["J_icon_l2"],
        "du_dc": results["du_dc"],
        "dc_coeffs": results["dc_coeffs"],
        "context_c_values": context["c_values"],
        "context_icon_mean": context["icon_mean"],
        "context_icon_std": context["icon_std"],
        "context_true": context["true"],
        "base_kinds": np.asarray(base_kinds),
    }

    if context_errors is not None:
        save_dict["context_cos_err"] = context_errors[0]
        save_dict["context_sin_err"] = context_errors[1]

    # Base spectra.
    for kind in base_kinds:
        item = results["base_spectra"][kind]
        save_dict[f"base_{kind}_k"] = item["k"]
        save_dict[f"base_{kind}_true"] = item["true"]
        save_dict[f"base_{kind}_icon_mean"] = item["icon_mean"]
        save_dict[f"base_{kind}_icon_std"] = item["icon_std"]

    # Split blocks.
    for group in ["true", "icon", "error"]:
        for block in ["cc", "cs", "sc", "ss"]:
            save_dict[f"blocks_{group}_{block}"] = results["split_blocks"][group][block]

    np.savez(os.path.join(fig_dir, "burgers_plot_only_diagnostics_results.npz"), **save_dict)


def plot_all_figures(results):
    # Main paper figures for the Burgers section.
    plot_prediction_example(results)
    plot_query_spectrum_with_true_error(results, pointwise_error="absolute")
    context_errors = plot_context_spectrum(results)
    plot_base_state_predictions(results)
    plot_base_state_spectra(results)
    plot_split_blocks(results)

    # Optional supplementary diagnostics. Uncomment if needed.
    # plot_singular_values(results)
    # plot_fixed_c_u0_jacobian_physical(results)
    # plot_c_sensitivity(results)
    # plot_base_state_viscosity_grid(results)

    save_numeric_results(results, context_errors=context_errors)




# =============================================================================
# 12. Paper-style Burgers diagnostics requested by the manuscript
#     This block overrides compute_all_diagnostics() and plot_all_figures()
#     defined above, so the main script below will generate the new figures.
# =============================================================================

# Evaluation grids.  Adjust these first if the script is too slow.
paper_c_values = np.asarray([0.002, 0.0035, 0.005, 0.01, 0.02, 0.035, 0.05, 0.07, 0.10])
paper_context_c_values = np.asarray([0.005, 0.01, 0.02, 0.035, 0.05])
paper_base_kinds = ("smooth", "steep", "near_shock")

paper_prediction_prompts_per_c = 10
paper_query_spectrum_prompts = 8
paper_base_contexts_per_c = 3
paper_feature_contexts_per_base = 1

paper_feature_map_c_value = 0.02
# Use a slightly smaller viscosity for the prescribed base-state prediction figure.
# This makes the three evolved profiles a bit sharper and closer to the low-viscosity
# Burgers regime emphasized in the spectra.
paper_base_prediction_c_value = 0.007
paper_feature_map_max_mode = 14
paper_query_max_mode = max_mode
paper_base_context_max_mode = 30

# Feature-map plotting convention.
# True and ICON maps always share one colorbar scale.  Error maps share another
# scale across all base states.  Set this to True if you want the error maps to
# use exactly the same scale as |A_true| and |A_ICON|.
PAPER_FEATURE_ERROR_SAME_SCALE = False
PAPER_FEATURE_CMAP = "magma"
PAPER_ERROR_CMAP = "cividis"
# Prediction errors are relative L2.
# Spectrum error panels use cumulative relative spectral error over modes,
# which is more stable than pointwise relative error when true high-frequency gains are tiny.


def _rel_l2_per_prompt(pred: np.ndarray, label: np.ndarray) -> np.ndarray:
    """Per-prompt relative L2 error for prediction plots."""
    target = np.asarray(label)[:, 0]          # [B, n_grid, 1]
    pred = np.asarray(pred)                  # [B, n_grid, 1]
    diff = pred - target
    diff_flat = diff.reshape(diff.shape[0], -1)
    target_flat = target.reshape(target.shape[0], -1)
    return np.linalg.norm(diff_flat, axis=1) / (np.linalg.norm(target_flat, axis=1) + 1e-12)




def _cumulative_relative_spectral_error(icon_cs: np.ndarray, true_cs: np.ndarray, eps: float = 1e-12):
    """
    Cumulative relative spectral errors over Fourier modes.

    Parameters
    ----------
    icon_cs, true_cs : np.ndarray
        Arrays of shape [num_c, num_modes, 2], with the last dimension
        storing [cos gain, sin gain].

    Returns
    -------
    cum_cos, cum_sin, cum_all : np.ndarray
        Arrays of shape [num_c, num_modes].  For example,

            cum_cos[c, K-1]
            = ||g_ICON_cos(c, 1:K) - g_true_cos(c, 1:K)||_2
              / (||g_true_cos(c, 1:K)||_2 + eps).

        The combined error uses both cos and sin components together.
    """
    icon_cs = np.asarray(icon_cs)
    true_cs = np.asarray(true_cs)
    diff2 = (icon_cs - true_cs) ** 2
    true2 = true_cs ** 2

    cum_cos = np.sqrt(np.cumsum(diff2[:, :, 0], axis=1)) / (
        np.sqrt(np.cumsum(true2[:, :, 0], axis=1)) + eps
    )
    cum_sin = np.sqrt(np.cumsum(diff2[:, :, 1], axis=1)) / (
        np.sqrt(np.cumsum(true2[:, :, 1], axis=1)) + eps
    )
    cum_all = np.sqrt(np.cumsum(np.sum(diff2, axis=2), axis=1)) / (
        np.sqrt(np.cumsum(np.sum(true2, axis=2), axis=1)) + eps
    )
    return cum_cos, cum_sin, cum_all


def compute_prediction_error_over_c(
    c_values=paper_c_values,
    prompts_per_c=paper_prediction_prompts_per_c,
    key_seed: int = 31415,
    representative_c: float = 0.02,
):
    """Compute the paper-style prediction error curve over hidden viscosity c."""
    rows = []
    for i, c_value in enumerate(c_values):
        raw_i, data_i, label_i = make_burgers_prompts_at_c(
            jax.random.PRNGKey(key_seed + i),
            c_value=float(c_value),
            num_prompts=prompts_per_c,
            pairs_per_prompt=num_pairs,
            fixed_query_u0=None,
        )
        pred_i = predict_batch(params, data_i)
        err_i = _rel_l2_per_prompt(np.asarray(pred_i), np.asarray(label_i))
        rows.append({
            "c": float(c_value),
            "err_mean": float(np.mean(err_i)),
            "err_std": float(np.std(err_i)),
            "err_all": err_i,
        })

    # Representative in-distribution prediction.
    raw_r, data_r, label_r = make_burgers_prompts_at_c(
        jax.random.PRNGKey(key_seed + 999),
        c_value=float(representative_c),
        num_prompts=1,
        pairs_per_prompt=num_pairs,
        fixed_query_u0=None,
    )
    pred_r = predict_batch(params, data_r)
    rel_r = float(_rel_l2_per_prompt(np.asarray(pred_r), np.asarray(label_r))[0])

    representative = {
        "c": float(raw_r.c[0]),
        "rel_l2": rel_r,
        "x": np.asarray(data_r.quest_qoi_k[0, 0, :, 0]),
        "u0": np.asarray(data_r.quest_cond_v[0, 0, :, 0]),
        "true": np.asarray(label_r[0, 0, :, 0]),
        "pred": np.asarray(pred_r[0, :, 0]),
    }

    return {
        "c_values": np.asarray([r["c"] for r in rows]),
        "err_mean": np.asarray([r["err_mean"] for r in rows]),
        "err_std": np.asarray([r["err_std"] for r in rows]),
        "err_all": [r["err_all"] for r in rows],
        "representative": representative,
    }


def compute_query_spectrum_and_error(
    c_values=paper_c_values,
    num_prompts_per_c: int = paper_query_spectrum_prompts,
    max_mode_local: int = paper_query_max_mode,
    key_seed: int = 27182,
):
    """
    Query-spectrum audit over several hidden viscosities.

    For each fixed c, generate several random prompts whose context and query
    share that c.  Then compare the ICON query Jacobian spectrum with the true
    Burgers tangent spectrum.  This produces:
      icon_mean[c, k, component], true_mean[c, k, component],
      pointwise error curves over k, and curve-level relative errors over c.

    The last component index is 0 for cos and 1 for sin.
    """
    k = np.arange(1, max_mode_local + 1)

    icon_mean_rows = []
    icon_std_rows = []
    true_mean_rows = []
    true_std_rows = []
    pred_rel_rows = []

    for ci, c_value in enumerate(c_values):
        raw, data, label = make_burgers_prompts_at_c(
            jax.random.PRNGKey(key_seed + ci),
            c_value=float(c_value),
            num_prompts=num_prompts_per_c,
            pairs_per_prompt=num_pairs,
            fixed_query_u0=None,
        )

        pred = predict_batch(params, data)
        pred_rel_rows.append(
            float(np.mean(_rel_l2_per_prompt(np.asarray(pred), np.asarray(label))))
        )

        icon_list = []
        true_list = []
        for pi in range(num_prompts_per_c):
            data_one = tree.tree_map(lambda x: x[pi], data)
            c_i = float(raw.c[pi])

            icon_gain_i = np.asarray(
                icon_mode_gain_by_jvp_fast(
                    params, data_one, max_mode=max_mode_local
                )
            )[:, 1:3]

            true_gain_i = np.asarray(
                true_mode_gain_by_jvp_u0_fast(
                    data_one.quest_cond_v.reshape(-1),
                    c_i,
                    max_mode=max_mode_local,
                )
            )[:, 1:3]

            icon_list.append(icon_gain_i)
            true_list.append(true_gain_i)

        icon_arr = np.stack(icon_list, axis=0)  # [P, K, 2]
        true_arr = np.stack(true_list, axis=0)

        icon_mean_rows.append(icon_arr.mean(axis=0))
        icon_std_rows.append(icon_arr.std(axis=0))
        true_mean_rows.append(true_arr.mean(axis=0))
        true_std_rows.append(true_arr.std(axis=0))

    icon_mean = np.stack(icon_mean_rows, axis=0)  # [C, K, 2]
    icon_std = np.stack(icon_std_rows, axis=0)
    true_mean = np.stack(true_mean_rows, axis=0)
    true_std = np.stack(true_std_rows, axis=0)

    abs_err = np.abs(icon_mean - true_mean)  # [C, K, 2]
    # Pointwise relative errors are saved for debugging only.  They are not
    # used in the main figure because high-frequency true gains can be tiny.
    rel_point_err = abs_err / np.maximum(np.abs(true_mean), 1e-6)

    # Curve-level relative errors for each c.  These are the terminal values
    # of the cumulative relative errors below.
    curve_err_cos_sin = np.linalg.norm(icon_mean - true_mean, axis=1) / (
        np.linalg.norm(true_mean, axis=1) + 1e-12
    )  # [C, 2]
    combined_curve_err = np.linalg.norm(icon_mean - true_mean, axis=(1, 2)) / (
        np.linalg.norm(true_mean, axis=(1, 2)) + 1e-12
    )  # [C]

    cum_rel_err_cos, cum_rel_err_sin, cum_rel_err_all = _cumulative_relative_spectral_error(
        icon_mean,
        true_mean,
    )

    return {
        "k": k,
        "c_values": np.asarray(c_values, dtype=float),
        "icon_mean": icon_mean,
        "icon_std": icon_std,
        "true_mean": true_mean,
        "true_std": true_std,
        "abs_err": abs_err,
        "rel_point_err": rel_point_err,
        "cum_rel_err_cos": cum_rel_err_cos,
        "cum_rel_err_sin": cum_rel_err_sin,
        "cum_rel_err_all": cum_rel_err_all,
        "curve_err_cos_sin": curve_err_cos_sin,
        "combined_curve_err": combined_curve_err,
        "prediction_rel_l2_over_c": np.asarray(pred_rel_rows),
    }

def compute_base_context_spectrum_and_error(
    base_kinds=paper_base_kinds,
    c_values=paper_context_c_values,
    contexts_per_c: int = paper_base_contexts_per_c,
    max_mode_local: int = paper_base_context_max_mode,
    key_seed: int = 16180,
):
    """
    For each prescribed base state, vary hidden viscosity c in the context and
    compare ICON context-conditioned tangent spectra with the true Burgers tangent spectra.
    """
    out = {}
    k = np.arange(1, max_mode_local + 1)

    for bi, kind in enumerate(base_kinds):
        uq = make_burgers_base_state(kind, n_grid=n_grid)
        icon_mean_rows, icon_std_rows, true_rows = [], [], []
        abs_curve_err_rows = []
        rel_err_rows = []

        for ci, c_value in enumerate(c_values):
            true_gain = np.asarray(
                true_mode_gain_by_jvp_u0_fast(
                    jnp.asarray(uq).reshape(-1),
                    float(c_value),
                    max_mode=max_mode_local,
                )
            )[:, 1:3]

            icon_contexts = []
            for r in range(contexts_per_c):
                data_one, _ = make_prompt_with_fixed_c_and_uq(
                    float(c_value),
                    uq,
                    key_seed=key_seed + 10000 * bi + 1000 * ci + r,
                )
                icon_gain = np.asarray(icon_mode_gain_by_jvp_fast(params, data_one, max_mode=max_mode_local))[:, 1:3]
                icon_contexts.append(icon_gain)

            icon_arr = np.stack(icon_contexts, axis=0)
            icon_mean = icon_arr.mean(axis=0)
            icon_std = icon_arr.std(axis=0)
            abs_curve_err = np.linalg.norm(icon_mean - true_gain, axis=0)
            rel_err = abs_curve_err / (
                np.linalg.norm(true_gain, axis=0) + 1e-12
            )

            icon_mean_rows.append(icon_mean)
            icon_std_rows.append(icon_std)
            true_rows.append(true_gain)
            abs_curve_err_rows.append(abs_curve_err)
            rel_err_rows.append(rel_err)

        icon_mean_stack = np.stack(icon_mean_rows, axis=0)  # [C, K, 2]
        icon_std_stack = np.stack(icon_std_rows, axis=0)
        true_stack = np.stack(true_rows, axis=0)

        cum_rel_err_cos, cum_rel_err_sin, cum_rel_err_all = _cumulative_relative_spectral_error(
            icon_mean_stack,
            true_stack,
        )

        out[kind] = {
            "k": k,
            "c_values": np.asarray(c_values, dtype=float),
            "icon_mean": icon_mean_stack,
            "icon_std": icon_std_stack,
            "true": true_stack,
            "abs_curve_err_cos_sin": np.stack(abs_curve_err_rows, axis=0),  # [C, 2]
            "rel_err_cos_sin": np.stack(rel_err_rows, axis=0),  # [C, 2]
            "cum_rel_err_cos": cum_rel_err_cos,  # [C, K]
            "cum_rel_err_sin": cum_rel_err_sin,  # [C, K]
            "cum_rel_err_all": cum_rel_err_all,  # [C, K]
            "combined_abs_curve_err": np.linalg.norm(icon_mean_stack - true_stack, axis=(1, 2)),
            "combined_rel_err": np.linalg.norm(icon_mean_stack - true_stack, axis=(1, 2)) / (
                np.linalg.norm(true_stack, axis=(1, 2)) + 1e-12
            ),
        }

    return out


def paired_fourier_basis_no_constant(n: int, max_mode_local: int) -> jnp.ndarray:
    """Real Fourier basis ordered as cos1, sin1, cos2, sin2, ... without the constant mode."""
    x = jnp.arange(n, dtype=jnp.float32) / n
    cols = []
    for k_mode in range(1, max_mode_local + 1):
        c_col = jnp.sqrt(2.0) * jnp.cos(2.0 * jnp.pi * k_mode * x)
        s_col = jnp.sqrt(2.0) * jnp.sin(2.0 * jnp.pi * k_mode * x)
        c_col = c_col / (jnp.linalg.norm(c_col) + 1e-12)
        s_col = s_col / (jnp.linalg.norm(s_col) + 1e-12)
        cols.extend([c_col, s_col])
    return jnp.stack(cols, axis=1)


def fourier_feature_matrix_from_jacobian(J_l2: jnp.ndarray, max_mode_local: int) -> jnp.ndarray:
    """A = Psi^T J Phi in paired Fourier ordering."""
    n_out, n_in = J_l2.shape
    Phi = paired_fourier_basis_no_constant(n_in, max_mode_local)
    Psi = paired_fourier_basis_no_constant(n_out, max_mode_local)
    return Psi.T @ J_l2 @ Phi


def _offdiag_ratio(A: np.ndarray, eps: float = 1e-12) -> float:
    diag = np.diag(np.diag(A))
    return float(np.linalg.norm(A - diag) / (np.linalg.norm(A) + eps))



def compute_base_state_predictions_paper(
    base_kinds=paper_base_kinds,
    c_value: float = paper_base_prediction_c_value,
    key_seed: int = 444,
):
    """Compute predictions for the three prescribed Burgers base states."""
    rows = []
    for i, kind in enumerate(base_kinds):
        uq = make_burgers_base_state(kind, n_grid=n_grid)
        raw, data, label = make_burgers_prompts_at_c(
            jax.random.PRNGKey(key_seed + i),
            c_value=float(c_value),
            num_prompts=1,
            pairs_per_prompt=num_pairs,
            fixed_query_u0=uq,
        )
        pred = predict_batch(params, data)
        rel_l2 = float(_rel_l2_per_prompt(np.asarray(pred), np.asarray(label))[0])
        rows.append({
            "kind": kind,
            "x": np.asarray(data.quest_qoi_k[0, 0, :, 0]),
            "u0": np.asarray(data.quest_cond_v[0, 0, :, 0]),
            "true": np.asarray(label[0, 0, :, 0]),
            "pred": np.asarray(pred[0, :, 0]),
            "c": float(raw.c[0]),
            "rel_l2": rel_l2,
        })
    return rows


def plot_base_state_predictions_paper(results):
    """Plot u0 and prediction for the three prescribed base states."""
    rows = results["base_state_predictions"]
    fig, axes = plt.subplots(len(rows), 2, figsize=(9.2, 3.0 * len(rows)), sharex=True)
    if len(rows) == 1:
        axes = axes[None, :]

    for i, row in enumerate(rows):
        axes[i, 0].plot(row["x"], row["u0"], "k-")
        axes[i, 0].set_ylabel(r"$u_0(x)$")
        axes[i, 0].grid(True)
        axes[i, 0].text(
            0.03, 0.88,
            row["kind"],
            transform=axes[i, 0].transAxes,
            ha="left", va="top",
            bbox=dict(facecolor="white", alpha=0.78, edgecolor="none"),
        )

        axes[i, 1].plot(row["x"], row["true"], "k-", label="true")
        axes[i, 1].plot(row["x"], row["pred"], "r--", label="ICON")
        axes[i, 1].set_ylabel(r"$u(T,x)$")
        axes[i, 1].grid(True)
        axes[i, 1].text(
            0.03, 0.88,
            rf"Rel-$L^2={row['rel_l2']:.2e}$",
            transform=axes[i, 1].transAxes,
            ha="left", va="top",
            bbox=dict(facecolor="white", alpha=0.78, edgecolor="none"),
        )
        if i == 0:
            axes[i, 1].legend()

    axes[-1, 0].set_xlabel(r"$x$")
    axes[-1, 1].set_xlabel(r"$x$")
    save_or_show(os.path.join(fig_dir, "burgers_base_state_predictions.png"))


def compute_feature_maps_and_error(
    base_kinds=paper_base_kinds,
    c_value: float = paper_feature_map_c_value,
    contexts_per_base: int = paper_feature_contexts_per_base,
    max_mode_local: int = paper_feature_map_max_mode,
    key_seed: int = 42424,
):
    """
    Compute Fourier feature maps A = Psi^T J Phi for true Burgers and ICON,
    plus the signed and absolute errors, for several base states.
    """
    rows = {}

    for bi, kind in enumerate(base_kinds):
        uq = make_burgers_base_state(kind, n_grid=n_grid)
        u0_flat = jnp.asarray(uq).reshape(-1)

        spec_true = true_burgers_jacobian_spectrum_u0(
            u0_flat,
            float(c_value),
            basis_size=2 * max_mode_local,
        )
        A_true = np.asarray(fourier_feature_matrix_from_jacobian(spec_true["J_l2"], max_mode_local))

        A_icon_contexts = []
        for r in range(contexts_per_base):
            data_one, _ = make_prompt_with_fixed_c_and_uq(
                float(c_value),
                uq,
                key_seed=key_seed + 1000 * bi + r,
            )
            spec_icon = icon_query_jacobian_spectrum(
                params,
                data_one,
                basis_size=2 * max_mode_local,
            )
            A_icon_contexts.append(
                np.asarray(fourier_feature_matrix_from_jacobian(spec_icon["J_l2"], max_mode_local))
            )

        A_icon = np.mean(np.stack(A_icon_contexts, axis=0), axis=0)
        A_err = A_icon - A_true

        rows[kind] = {
            "A_true": A_true,
            "A_icon": A_icon,
            "A_err": A_err,
            "abs_true": np.abs(A_true),
            "abs_icon": np.abs(A_icon),
            "abs_err": np.abs(A_err),
            "abs_frob_err": float(np.linalg.norm(A_err)),
            "rel_err_map": np.abs(A_err) / (np.linalg.norm(A_true) + 1e-12),
            "rel_frob_err": float(np.linalg.norm(A_err) / (np.linalg.norm(A_true) + 1e-12)),
            "roff_true": _offdiag_ratio(A_true),
            "roff_icon": _offdiag_ratio(A_icon),
            "c": float(c_value),
            "max_mode": int(max_mode_local),
        }

    return rows



def plot_prediction_error_over_c_paper(results):
    """Plot only the relative prediction error over hidden viscosity c."""
    from plotting.regimes import (style_parameter_curves, style_parameter_sweep, add_regime_key, comparison_filename)
    item = results["prediction_error"]
    c_values = item["c_values"]
    err_mean = item["err_mean"]
    err_std = item["err_std"]

    plt.figure(figsize=(6.4, 4.1))
    # plt.errorbar(c_values, err_mean, yerr=err_std, fmt="o-", capsize=3, label="mean ± std")
    plt.plot(c_values, err_mean, "o-", label="mean")
    plt.axvspan(c_range[0], c_range[1], alpha=0.18, label="training range")
    plt.yscale("log")
    plt.xlabel(r"hidden viscosity $c$")
    plt.ylabel(r"relative $L^2$ prediction error")
    plt.grid(True, which="both")
    for regime_ax in np.asarray(plt.gca(), dtype=object).ravel():
        style_parameter_sweep(regime_ax, 'burgers')
    plt.legend()
    add_regime_key(plt.gcf())
    save_or_show(comparison_filename(os.path.join(fig_dir, "burgers_relative_prediction_error_over_hidden_viscosity.png")))



def plot_query_spectrum_and_error_paper(results):
    """
    Plot query spectra for multiple hidden viscosities.

    Layout:
      - top-left/top-right: ICON gains only for cos/sin, for every c;
      - bottom-left/bottom-right: cumulative relative spectral errors over modes.

    The true Burgers spectra are used internally as the reference for the
    cumulative relative errors, but they are not plotted in the gain panels.
    This avoids visual clutter while still quantifying agreement with the
    solver-based tangent spectrum.
    """
    from plotting.regimes import (style_parameter_curves, style_parameter_sweep, add_regime_key, comparison_filename)
    item = results["query_spectrum"]
    k = item["k"]
    c_values = item["c_values"]
    icon_mean = item["icon_mean"]      # [C, K, 2]
    icon_std = item["icon_std"]        # [C, K, 2]
    cum_cos = item["cum_rel_err_cos"]  # [C, K]
    cum_sin = item["cum_rel_err_sin"]  # [C, K]

    fig, axes = plt.subplots(2, 2, figsize=(10.6, 7.0), sharex="col")
    ax_gain_cos, ax_gain_sin = axes[0]
    ax_err_cos, ax_err_sin = axes[1]

    gain_axes = [ax_gain_cos, ax_gain_sin]
    err_axes = [ax_err_cos, ax_err_sin]
    err_data = [cum_cos, cum_sin]
    names = ["cos", "sin"]
    markers = ["o", "s"]

    for ci, c_value in enumerate(c_values):
        for j, name in enumerate(names):
            ax = gain_axes[j]
            line, = ax.plot(
                k,
                icon_mean[ci, :, j],
                marker=markers[j],
                linestyle="-",
                ms=3,
                label=f"{c_value:g}" if j == 1 else None,
            )
            color = line.get_color()

            # Optional uncertainty over prompts.  Kept light so the main curves remain readable.
            ax.fill_between(
                k,
                np.maximum(icon_mean[ci, :, j] - icon_std[ci, :, j], 0.0),
                icon_mean[ci, :, j] + icon_std[ci, :, j],
                color=color,
                alpha=0.05,
                linewidth=0.0,
            )

            err_axes[j].plot(
                k,
                err_data[j][ci],
                marker=markers[j],
                linestyle="-",
                ms=3,
                color=color,
            )

    for ax, name in zip(gain_axes, names):
        ax.axvline(n_modes, linestyle="--", color="k")
        ax.grid(True, which="both")
        ax.set_ylabel("ICON Jacobian gain")
        ax.text(0.04, 0.90, name, transform=ax.transAxes,
                ha="left", va="top",
                bbox=dict(facecolor="white", alpha=0.78, edgecolor="none"))

    for ax, name in zip(err_axes, names):
        ax.axvline(n_modes, linestyle="--", color="k")
        ax.grid(True, which="both")
        ax.set_xlabel(r"Fourier cutoff $K$")
        ax.set_ylabel("cumulative relative error")
        ax.text(0.04, 0.90, name, transform=ax.transAxes,
                ha="left", va="top",
                bbox=dict(facecolor="white", alpha=0.78, edgecolor="none"))

    style_parameter_curves(axes, c_values, 'burgers')
    ax_gain_sin.legend(title=r"hidden $c$", ncol=2, fontsize=9)
    add_regime_key(plt.gcf())
    save_or_show(comparison_filename(os.path.join(fig_dir, "burgers_query_spectrum_over_c_with_cumulative_relative_error.png")))

def plot_base_context_spectrum_and_error_paper(results):
    """
    Plot base-state context spectra and cumulative relative errors.

    For each prescribed base state:
      - first two columns: ICON gains only for cos/sin and several c values;
      - last two columns: cumulative relative spectral error up to Fourier cutoff K.

    The true Burgers spectra are used internally as the reference for the cumulative
    errors, but are not overlaid in the gain panels to avoid visual clutter.
    """
    base_context = results["base_context_spectrum"]
    kinds = list(base_context.keys())

    fig, axes = plt.subplots(len(kinds), 4, figsize=(17.0, 3.55 * len(kinds)), sharex="col")
    if len(kinds) == 1:
        axes = axes[None, :]

    for i, kind in enumerate(kinds):
        item = base_context[kind]
        k = item["k"]
        c_values = item["c_values"]
        icon_mean = item["icon_mean"]
        icon_std = item["icon_std"]
        cum_cos = item["cum_rel_err_cos"]
        cum_sin = item["cum_rel_err_sin"]

        ax_gain_cos, ax_gain_sin, ax_err_cos, ax_err_sin = axes[i]

        for ci, c_value in enumerate(c_values):
            # Cos gain.
            line_cos, = ax_gain_cos.plot(
                k,
                icon_mean[ci, :, 0],
                "o-",
                ms=3,
                label=f"{c_value:g}",
            )
            color_cos = line_cos.get_color()
            ax_gain_cos.fill_between(
                k,
                np.maximum(icon_mean[ci, :, 0] - icon_std[ci, :, 0], 0.0),
                icon_mean[ci, :, 0] + icon_std[ci, :, 0],
                color=color_cos,
                alpha=0.05,
                linewidth=0.0,
            )
            ax_err_cos.plot(k, cum_cos[ci], "o-", ms=3, color=color_cos)

            # Sin gain.
            line_sin, = ax_gain_sin.plot(
                k,
                icon_mean[ci, :, 1],
                "s-",
                ms=3,
                label=f"{c_value:g}",
            )
            color_sin = line_sin.get_color()
            ax_gain_sin.fill_between(
                k,
                np.maximum(icon_mean[ci, :, 1] - icon_std[ci, :, 1], 0.0),
                icon_mean[ci, :, 1] + icon_std[ci, :, 1],
                color=color_sin,
                alpha=0.05,
                linewidth=0.0,
            )
            ax_err_sin.plot(k, cum_sin[ci], "s-", ms=3, color=color_sin)

        panel_specs = [
            (ax_gain_cos, "cos gain", "ICON Jacobian gain", r"Fourier mode $k$"),
            (ax_gain_sin, "sin gain", "ICON Jacobian gain", r"Fourier mode $k$"),
            (ax_err_cos, "cos accumulated error", "cumulative relative error", r"Fourier cutoff $K$"),
            (ax_err_sin, "sin accumulated error", "cumulative relative error", r"Fourier cutoff $K$"),
        ]

        for ax, label, ylabel, xlabel in panel_specs:
            ax.axvline(n_modes, linestyle="--", color="k")
            ax.grid(True, which="both")
            ax.set_xlabel(xlabel)
            ax.set_ylabel(ylabel)
            ax.text(
                0.04,
                0.90,
                f"{kind}, {label}",
                transform=ax.transAxes,
                ha="left",
                va="top",
                bbox=dict(facecolor="white", alpha=0.78, edgecolor="none"),
            )

        if i == 0:
            ax_gain_sin.legend(title=r"hidden $c$", ncol=2, fontsize=9)

    save_or_show(os.path.join(fig_dir, "burgers_base_state_context_spectrum_with_cumulative_relative_error.png"))



def plot_feature_maps_with_error_paper(
    results,
    feature_cmap: str = PAPER_FEATURE_CMAP,
    error_cmap: str = PAPER_ERROR_CMAP,
    error_same_scale: bool = PAPER_FEATURE_ERROR_SAME_SCALE,
):
    """
    Plot |A_true|, |A_ICON|, and an absolute error map for each base state.

    Colorbar convention:
      - |A_true| and |A_ICON| share exactly the same scale over all rows.
      - Absolute error maps share one common scale over all rows.
      - If error_same_scale=True, the absolute error maps use the feature-map scale.
    """
    fmap = results["feature_maps"]
    kinds = list(fmap.keys())

    feature_vmax = max(
        max(float(np.max(fmap[k]["abs_true"])), float(np.max(fmap[k]["abs_icon"])))
        for k in kinds
    ) + 1e-12
    abs_err_vmax_auto = max(float(np.max(fmap[k]["abs_err"])) for k in kinds) + 1e-12
    err_vmax = feature_vmax if error_same_scale else abs_err_vmax_auto

    fig, axes = plt.subplots(len(kinds), 3, figsize=(12.2, 3.45 * len(kinds)), constrained_layout=True)
    if len(kinds) == 1:
        axes = axes[None, :]

    main_im = None
    err_im = None
    for i, kind in enumerate(kinds):
        row = fmap[kind]
        panels = [
            (row["abs_true"], "true", feature_cmap, feature_vmax),
            (row["abs_icon"], "ICON", feature_cmap, feature_vmax),
            (row["abs_err"], "absolute error", error_cmap, err_vmax),
        ]

        for j, (M, label, cmap, vmax) in enumerate(panels):
            ax = axes[i, j]
            im = ax.imshow(
                M,
                origin="lower",
                aspect="auto",
                cmap=cmap,
                vmin=0.0,
                vmax=vmax,
                interpolation="nearest",
            )
            if j < 2:
                main_im = im
            else:
                err_im = im
            ax.set_xlabel("input Fourier index")
            ax.set_ylabel("output Fourier index")
            ax.text(
                0.03, 0.94,
                f"{kind}, {label}",
                transform=ax.transAxes,
                ha="left", va="top",
                bbox=dict(facecolor="white", alpha=0.80, edgecolor="none"),
            )

        axes[i, 1].text(
            0.03, 0.06,
            rf"$r_{{off}}={row['roff_icon']:.2f}$",
            transform=axes[i, 1].transAxes,
            ha="left", va="bottom",
            bbox=dict(facecolor="white", alpha=0.80, edgecolor="none"),
        )
        axes[i, 2].text(
            0.03, 0.06,
            rf"abs. Fro. err $={row['abs_frob_err']:.2e}$",
            transform=axes[i, 2].transAxes,
            ha="left", va="bottom",
            bbox=dict(facecolor="white", alpha=0.80, edgecolor="none"),
        )

    fig.colorbar(main_im, ax=axes[:, :2].ravel().tolist(), shrink=0.88, label=r"$|A|$")
    fig.colorbar(err_im, ax=axes[:, 2].ravel().tolist(), shrink=0.88, label=r"$|A_{ICON}-A_{true}|$")

    fname = "burgers_fourier_feature_maps_true_icon_absolute_error.png"
    if error_same_scale:
        fname = "burgers_fourier_feature_maps_true_icon_absolute_error_same_scale.png"
    if save_figs:
        plt.savefig(os.path.join(fig_dir, fname), dpi=240, bbox_inches="tight")
    plt.show()



def _split_pair_ordered_fourier_matrix(A: np.ndarray):
    """
    Split a paired real-Fourier matrix into cos/sin blocks.

    The paired ordering is
        cos(1), sin(1), cos(2), sin(2), ...

    Returned blocks use rows = output modes and columns = input modes:
        cc: input cos -> output cos
        cs: input sin -> output cos
        sc: input cos -> output sin
        ss: input sin -> output sin
    """
    A = np.asarray(A)
    return {
        "cc": A[0::2, 0::2],
        "cs": A[0::2, 1::2],
        "sc": A[1::2, 0::2],
        "ss": A[1::2, 1::2],
    }


def plot_feature_maps_split_cos_sin_paper(
    results,
    feature_cmap: str = PAPER_FEATURE_CMAP,
    error_cmap: str = PAPER_ERROR_CMAP,
    error_same_scale: bool = PAPER_FEATURE_ERROR_SAME_SCALE,
):
    """
    Split the Fourier feature map into the four real Fourier blocks:
        cos -> cos, sin -> cos, cos -> sin, sin -> sin.

    For each Burgers base state, this saves a separate figure with layout

        row 1: |A_true| blocks
        row 2: |A_ICON| blocks
        row 3: |A_ICON - A_true| blocks

    This is easier to interpret than the paired-index matrix because it
    separates sine and cosine input/output channels explicitly.
    """
    fmap = results["feature_maps"]
    kinds = list(fmap.keys())
    block_order = [
        ("cc", r"$\cos\to\cos$"),
        ("cs", r"$\sin\to\cos$"),
        ("sc", r"$\cos\to\sin$"),
        ("ss", r"$\sin\to\sin$"),
    ]

    # Use a common scale across all split true/ICON blocks and a separate
    # common scale across all split error blocks.
    feature_vmax = 1e-12
    err_vmax_auto = 1e-12
    split_cache = {}

    for kind in kinds:
        row = fmap[kind]
        true_blocks = _split_pair_ordered_fourier_matrix(row["A_true"])
        icon_blocks = _split_pair_ordered_fourier_matrix(row["A_icon"])
        err_blocks = _split_pair_ordered_fourier_matrix(row["A_err"])

        split_cache[kind] = {
            "true": true_blocks,
            "icon": icon_blocks,
            "error": err_blocks,
        }

        for key, _ in block_order:
            feature_vmax = max(
                feature_vmax,
                float(np.max(np.abs(true_blocks[key]))),
                float(np.max(np.abs(icon_blocks[key]))),
            )
            err_vmax_auto = max(err_vmax_auto, float(np.max(np.abs(err_blocks[key]))))

    err_vmax = feature_vmax if error_same_scale else err_vmax_auto

    for kind in kinds:
        row = fmap[kind]
        blocks = split_cache[kind]

        fig, axes = plt.subplots(
            3, 4,
            figsize=(13.8, 8.2),
            constrained_layout=True,
            sharex=True,
            sharey=True,
        )

        row_specs = [
            ("true", "true", feature_cmap, feature_vmax, r"$|A|$"),
            ("icon", "ICON", feature_cmap, feature_vmax, r"$|A|$"),
            ("error", "absolute error", error_cmap, err_vmax, r"$|A_{\rm ICON}-A_{\rm true}|$"),
        ]

        main_im = None
        err_im = None
        for i, (group, row_label, cmap, vmax, _) in enumerate(row_specs):
            for j, (block_key, block_label) in enumerate(block_order):
                ax = axes[i, j]
                M = np.abs(blocks[group][block_key])
                im = ax.imshow(
                    M,
                    origin="lower",
                    aspect="auto",
                    cmap=cmap,
                    vmin=0.0,
                    vmax=vmax,
                    interpolation="nearest",
                )
                if group == "error":
                    err_im = im
                else:
                    main_im = im

                if i == 0:
                    ax.set_title(block_label, fontsize=11)
                if j == 0:
                    ax.set_ylabel(f"{row_label}\noutput mode")
                else:
                    ax.set_ylabel("")
                if i == 2:
                    ax.set_xlabel("input mode")

                ax.text(
                    0.04, 0.93,
                    row_label if j == 0 else "",
                    transform=ax.transAxes,
                    ha="left", va="top",
                    fontsize=9,
                    bbox=dict(facecolor="white", alpha=0.78, edgecolor="none"),
                )

        axes[1, 0].text(
            0.04, 0.08,
            rf"$r_{{off}}={row['roff_icon']:.2f}$",
            transform=axes[1, 0].transAxes,
            ha="left", va="bottom",
            fontsize=9,
            bbox=dict(facecolor="white", alpha=0.80, edgecolor="none"),
        )
        axes[2, 0].text(
            0.04, 0.08,
            rf"Rel-F={row['rel_frob_err']:.2e}",
            transform=axes[2, 0].transAxes,
            ha="left", va="bottom",
            fontsize=9,
            bbox=dict(facecolor="white", alpha=0.80, edgecolor="none"),
        )

        fig.colorbar(main_im, ax=axes[:2, :].ravel().tolist(), shrink=0.82, label=r"$|A|$")
        fig.colorbar(err_im, ax=axes[2, :].ravel().tolist(), shrink=0.82, label=r"$|A_{\rm ICON}-A_{\rm true}|$")

        filename = f"burgers_fourier_feature_blocks_{kind}_true_icon_absolute_error.png"
        if error_same_scale:
            filename = f"burgers_fourier_feature_blocks_{kind}_true_icon_absolute_error_same_scale.png"
        if save_figs:
            fig.savefig(os.path.join(fig_dir, filename), dpi=240, bbox_inches="tight")
        plt.show()


def save_paper_numeric_results(results):
    """Save the arrays behind the new figures."""
    pred = results["prediction_error"]
    query = results["query_spectrum"]
    save_dict = {
        "prediction_c_values": pred["c_values"],
        "prediction_err_mean": pred["err_mean"],
        "prediction_err_std": pred["err_std"],
        "query_k": query["k"],
        "query_icon_mean": query["icon_mean"],
        "query_icon_std": query["icon_std"],
        "query_true_mean": query["true_mean"],
        "query_true_std": query["true_std"],
        "query_abs_err": query["abs_err"],
        "query_rel_point_err": query["rel_point_err"],
        "query_cum_rel_err_cos": query["cum_rel_err_cos"],
        "query_cum_rel_err_sin": query["cum_rel_err_sin"],
        "query_cum_rel_err_all": query["cum_rel_err_all"],
        "query_curve_err_cos_sin": query["curve_err_cos_sin"],
        "query_combined_curve_err": np.asarray(query["combined_curve_err"]),
    }

    for row in results.get("base_state_predictions", []):
        kind = row["kind"]
        save_dict[f"base_prediction_{kind}_x"] = row["x"]
        save_dict[f"base_prediction_{kind}_u0"] = row["u0"]
        save_dict[f"base_prediction_{kind}_true"] = row["true"]
        save_dict[f"base_prediction_{kind}_pred"] = row["pred"]
        save_dict[f"base_prediction_{kind}_rel_l2"] = np.asarray(row["rel_l2"])

    for kind, item in results["base_context_spectrum"].items():
        save_dict[f"base_context_{kind}_k"] = item["k"]
        save_dict[f"base_context_{kind}_c_values"] = item["c_values"]
        save_dict[f"base_context_{kind}_icon_mean"] = item["icon_mean"]
        save_dict[f"base_context_{kind}_icon_std"] = item["icon_std"]
        save_dict[f"base_context_{kind}_true"] = item["true"]
        save_dict[f"base_context_{kind}_abs_curve_err_cos_sin"] = item["abs_curve_err_cos_sin"]
        save_dict[f"base_context_{kind}_combined_abs_curve_err"] = item["combined_abs_curve_err"]
        save_dict[f"base_context_{kind}_rel_err_cos_sin"] = item["rel_err_cos_sin"]
        save_dict[f"base_context_{kind}_cum_rel_err_cos"] = item["cum_rel_err_cos"]
        save_dict[f"base_context_{kind}_cum_rel_err_sin"] = item["cum_rel_err_sin"]
        save_dict[f"base_context_{kind}_cum_rel_err_all"] = item["cum_rel_err_all"]
        save_dict[f"base_context_{kind}_combined_rel_err"] = item["combined_rel_err"]

    for kind, item in results["feature_maps"].items():
        save_dict[f"feature_{kind}_A_true"] = item["A_true"]
        save_dict[f"feature_{kind}_A_icon"] = item["A_icon"]
        save_dict[f"feature_{kind}_A_err"] = item["A_err"]
        save_dict[f"feature_{kind}_abs_true"] = item["abs_true"]
        save_dict[f"feature_{kind}_abs_icon"] = item["abs_icon"]
        save_dict[f"feature_{kind}_abs_err"] = item["abs_err"]
        save_dict[f"feature_{kind}_rel_err_map"] = item["rel_err_map"]
        save_dict[f"feature_{kind}_abs_frob_err"] = np.asarray(item["abs_frob_err"])
        save_dict[f"feature_{kind}_rel_frob_err"] = np.asarray(item["rel_frob_err"])
        save_dict[f"feature_{kind}_roff_true"] = np.asarray(item["roff_true"])
        save_dict[f"feature_{kind}_roff_icon"] = np.asarray(item["roff_icon"])

        # Split cos/sin block matrices in paired real-Fourier ordering:
        # cc: cos->cos, cs: sin->cos, sc: cos->sin, ss: sin->sin.
        for block_name, block_mat in _split_pair_ordered_fourier_matrix(item["A_true"]).items():
            save_dict[f"feature_{kind}_true_block_{block_name}"] = block_mat
        for block_name, block_mat in _split_pair_ordered_fourier_matrix(item["A_icon"]).items():
            save_dict[f"feature_{kind}_icon_block_{block_name}"] = block_mat
        for block_name, block_mat in _split_pair_ordered_fourier_matrix(item["A_err"]).items():
            save_dict[f"feature_{kind}_err_block_{block_name}"] = block_mat

    np.savez(os.path.join(fig_dir, "burgers_paper_style_diagnostics_results.npz"), **save_dict)


def compute_all_diagnostics():
    print("\n[paper] Computing prediction error over hidden viscosity...")
    prediction_error = compute_prediction_error_over_c()

    print("[paper] Computing query spectrum and true-reference errors...")
    query_spectrum = compute_query_spectrum_and_error()

    print("[paper] Computing base-state predictions...")
    base_state_predictions = compute_base_state_predictions_paper()

    print("[paper] Computing base-state context spectra and errors...")
    base_context_spectrum = compute_base_context_spectrum_and_error()

    print("[paper] Computing Fourier feature maps and errors...")
    feature_maps = compute_feature_maps_and_error()

    return {
        "prediction_error": prediction_error,
        "query_spectrum": query_spectrum,
        "base_state_predictions": base_state_predictions,
        "base_context_spectrum": base_context_spectrum,
        "feature_maps": feature_maps,
    }


def plot_all_figures(results):
    plot_prediction_error_over_c_paper(results)
    plot_base_state_predictions_paper(results)
    plot_query_spectrum_and_error_paper(results)
    plot_base_context_spectrum_and_error_paper(results)
    # Combined paired-index feature map, kept for backward compatibility.
    plot_feature_maps_with_error_paper(results)

    # Split cos/sin Fourier blocks: cc, cs, sc, ss.
    plot_feature_maps_split_cos_sin_paper(results)

    save_paper_numeric_results(results)


# =============================================================================
# 12. Main
# =============================================================================

if __name__ == "__main__":
    os.makedirs(fig_dir, exist_ok=True)
    print("JAX devices:", jax.devices())
    print("Loaded checkpoint:", loaded_ckpt_path)
    final_results = compute_all_diagnostics()
    plot_all_figures(final_results)
    print("Saved figures and arrays to:", fig_dir)
